# 01 — Time Series Understanding and Exploration

## 1. Study Context and Univariate Forecasting Objective

This repository is being developed as a standalone scientific study of the `nottem` dataset to establish a reproducible reference workflow for **univariate time-series forecasting** before any forecasting capability is implemented in Atlas DataFlow.

The study treats forecasting as a temporally ordered prediction problem rather than as ordinary tabular continuous regression. The time axis is part of the scientific identity of each observation, so all later preparation, validation, model selection, final evaluation, and inference decisions must preserve temporal causality and prevent future observations from influencing earlier fitted operations or evaluations.

The objective of this notebook is to build the evidence needed to define a defensible forecasting contract from the source series itself. The study is intentionally constrained to a single endogenous target series: forecasting must be based on its observed history, without introducing exogenous predictors that are not part of the original source.

The existing repository was adapted from a continuous-regression study. Any inherited regression-specific code, contracts, metrics, models, artifacts, documentation, or assumptions are therefore structural references only and are not evidence for the scientific design of this forecasting study.

At this stage, the following remain intentionally open and must be authenticated or decided only in their dedicated sections:

- source identity and reproducible Python acquisition;
- source time-series representation and canonical temporal index;
- observed frequency, temporal coverage, continuity, and target unit;
- trend, seasonality, lag dependence, stationarity signals, anomalies, and structural changes;
- forecast horizon and final-holdout boundary;
- backtesting design and forecasting-origin semantics;
- baseline definitions and model families;
- primary and secondary forecasting metrics; and
- the exact machine-readable forecasting contract and downstream artifact semantics.

This notebook is exploratory and contractual only. It does not implement Atlas integration, fit forecasting models, perform model selection, open a final holdout, or define downstream production behavior.

## 2. Dataset Source and Python Acquisition

In [1]:
from IPython.display import display
import json
from pathlib import Path

import pandas as pd

import scripts.download_data as download_data_module
import scripts.forecasting_exploration_handoff as handoff_module
import scripts.project_context as project_context_module
from scripts.download_data import acquire_rdataset
from scripts.project_context import get_project_context


DATASET_NAME = "nottem"
R_PACKAGE = "datasets"
PROJECT = get_project_context()

for critical_module in (
    download_data_module,
    project_context_module,
    handoff_module,
):
    module_file = getattr(critical_module, "__file__", None)
    if module_file is None:
        raise RuntimeError(
            f"Critical project module has no filesystem origin: "
            f"{critical_module.__name__}."
        )
    try:
        Path(module_file).resolve().relative_to(PROJECT.root)
    except ValueError as exc:
        raise RuntimeError(
            "Project scripts are being imported outside the active checkout. "
            "Restart the kernel from the project checkout or use an editable "
            "installation of this repository."
        ) from exc

RAW_DATA_DIR = PROJECT.path("data", "raw", DATASET_NAME)

acquisition = acquire_rdataset(
    dataset_name=DATASET_NAME,
    package=R_PACKAGE,
    destination=RAW_DATA_DIR,
    project_root=PROJECT.root,
)

data_path = acquisition.require_one_file("dataset.csv")
metadata_path = acquisition.require_one_file("metadata.json")
documentation_path = acquisition.require_one_file("documentation.txt")

for acquired_path in (
    data_path,
    metadata_path,
    documentation_path,
):
    try:
        acquired_path.resolve().relative_to(PROJECT.root)
    except ValueError as exc:
        raise RuntimeError(
            f"Acquired file escaped the authenticated project root: "
            f"{acquired_path.name}."
        ) from exc

source_data = pd.read_csv(data_path)
source_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

print(f"Source type: {acquisition.source_kind}")
print(f"Source reference: {acquisition.source_reference}")
print(f"Destination: {acquisition.display_destination}")
print(f"Materialized files: {list(acquisition.relative_files)}")
print(f"Source title: {source_metadata['title']}")
print(f"Raw shape: {source_data.shape}")
print(f"Raw columns: {source_data.columns.tolist()}")
print(f"Documentation file: {documentation_path.name}")

display(source_data.head())

RuntimeError: Project scripts are being imported outside the active checkout. Restart the kernel from the project checkout or use an editable installation of this repository.

## 3. Source Time-Series Representation and Temporal Observation Identity

The scientific source is the R object `datasets::nottem`, documented as a **time-series object** containing average air temperatures at Nottingham Castle in degrees Fahrenheit. Its title identifies the observations as monthly averages. The source is therefore scientifically univariate: one temporally ordered scalar temperature measurement is associated with each source time coordinate, and no exogenous predictor series is part of the original dataset.

The Python acquisition layer materializes that source as a two-column table named `time` and `value`. This tabular transport representation does **not** redefine the problem as tabular regression and does not make `time` an ordinary predictive feature. Instead:

- `time` is the source-series temporal coordinate carried through the Rdatasets representation;
- `value` is the single observed endogenous measurement;
- row order preserves the source-series order;
- one row represents one monthly average air-temperature observation at Nottingham Castle; and
- there are no source exogenous predictors to add to the forecasting input contract.

The acquired `time` values are numeric fractional-year coordinates rather than authenticated calendar timestamps. They must therefore be preserved as source coordinates at this stage, not silently converted to arbitrary dates. The source row position together with the raw `time` coordinate provides the provisional observation identity until the canonical monthly index is reconstructed and validated in the next section.

The R `ts` semantics also do not imply a timezone-bearing instant. A monthly average is a calendar-period observation, not an event timestamp. Exact calendar labeling, frequency verification, start/end coverage, continuity, and timezone applicability are intentionally deferred to the canonical time-index reconstruction step.

This distinction is contractual: the **target value** is the observed temperature measurement, while forecast horizon, forecast origin, and allowed historical context are separate forecasting concepts that remain undefined at this point.

In [ ]:
EXPECTED_SOURCE_COLUMNS = ("time", "value")
EXPECTED_SOURCE_REFERENCE = "datasets::nottem"

observed_columns = tuple(source_data.columns)

if observed_columns != EXPECTED_SOURCE_COLUMNS:
    raise ValueError(
        "Unexpected source representation: "
        f"expected columns {EXPECTED_SOURCE_COLUMNS}, observed {observed_columns}."
    )

if source_metadata.get("source_reference") != EXPECTED_SOURCE_REFERENCE:
    raise ValueError(
        "Unexpected source identity: "
        f"expected {EXPECTED_SOURCE_REFERENCE!r}, "
        f"observed {source_metadata.get('source_reference')!r}."
    )

if not pd.api.types.is_numeric_dtype(source_data["time"]):
    raise TypeError("The raw R time coordinate must remain numeric at this stage.")

if not pd.api.types.is_numeric_dtype(source_data["value"]):
    raise TypeError("The source value column must be numeric.")

source_representation = pd.Series(
    {
        "scientific_source": "datasets::nottem",
        "source_object_semantics": "univariate R time-series object",
        "python_materialization": "two-column table: time, value",
        "raw_time_role": "source temporal coordinate; not a predictive feature",
        "raw_time_representation": "numeric fractional-year coordinate",
        "observed_value_role": "single endogenous temperature measurement",
        "temporal_observation_unit": (
            "one monthly average air-temperature observation at Nottingham Castle"
        ),
        "source_exogenous_predictors": 0,
        "canonical_calendar_index": "not reconstructed yet",
        "timezone_semantics": "not assigned; applicability still to be confirmed",
    },
    name="source representation",
)

display(source_representation.to_frame())
display(source_data.loc[:, list(EXPECTED_SOURCE_COLUMNS)].head())

## 4. Canonical Time Index Reconstruction, Frequency, and Temporal Coverage

The authenticated source semantics identify `datasets::nottem` as monthly average temperatures covering 1920–1939. The acquired Python table carries the original R time-series coordinate as fractional years, so this section converts that source coordinate into an explicit calendar-period index without inventing day-level or timezone-bearing timestamps.

A **monthly `pandas.PeriodIndex`** is the canonical temporal representation for this study. A calendar month is the scientific observation period, whereas a `DatetimeIndex` would require an arbitrary day or instant that is not part of the source semantics. Timezone assignment is therefore not applicable to the canonical index.

The reconstruction follows the R `ts` coordinate semantics for a monthly series:

- the integer component of `time` identifies the calendar year;
- the fractional component is converted to a zero-based month position using 12 source periods per year;
- the reconstructed periods must align with the authenticated source coverage from `1920-01` through `1939-12`;
- the resulting canonical index must contain exactly 240 monthly periods; and
- the numerical fractional-year representation is accepted only when it is sufficiently close to the expected monthly grid.

The source frequency is therefore frozen as **monthly, 12 observations per year**. This frequency defines the calendar spacing of the observations; it does not by itself prove seasonal dependence or justify a particular forecasting horizon. Seasonal strength and lag dependence remain subjects of later exploratory sections.

The canonical series constructed here preserves the acquired values exactly and changes only their temporal labeling. No resampling, interpolation, aggregation, smoothing, differencing, seasonal adjustment, imputation, or forecasting feature engineering is performed.

Regularity is validated here only as required to authenticate the reconstruction and coverage. Dedicated diagnostics for missing periods, duplicate timestamps, ordering integrity, and related data-quality conditions remain in their later sections.

In [ ]:
import numpy as np


EXPECTED_FREQUENCY = "M"
SOURCE_PERIODS_PER_YEAR = 12
EXPECTED_START_PERIOD = pd.Period("1920-01", freq=EXPECTED_FREQUENCY)
EXPECTED_END_PERIOD = pd.Period("1939-12", freq=EXPECTED_FREQUENCY)
EXPECTED_OBSERVATIONS = 240
TIME_COORDINATE_TOLERANCE = 1e-8

raw_time = source_data["time"].to_numpy(dtype=float, copy=True)

source_year = np.floor(raw_time).astype(int)
source_month_position = (raw_time - source_year) * SOURCE_PERIODS_PER_YEAR
source_month_offset = np.rint(source_month_position).astype(int)

max_coordinate_residual = float(
    np.max(np.abs(source_month_position - source_month_offset))
)

if max_coordinate_residual > TIME_COORDINATE_TOLERANCE:
    raise ValueError(
        "The raw time coordinate is not aligned with authenticated monthly "
        "R ts semantics within tolerance: "
        f"max residual={max_coordinate_residual:.3e}."
    )

if np.any(
    (source_month_offset < 0)
    | (source_month_offset >= SOURCE_PERIODS_PER_YEAR)
):
    raise ValueError("The raw time coordinate produced an invalid month offset.")

canonical_period_index = pd.PeriodIndex.from_fields(
    year=source_year,
    month=source_month_offset + 1,
    freq=EXPECTED_FREQUENCY,
).rename("period")

expected_period_index = pd.period_range(
    start=EXPECTED_START_PERIOD,
    end=EXPECTED_END_PERIOD,
    freq=EXPECTED_FREQUENCY,
    name="period",
)

if len(canonical_period_index) != EXPECTED_OBSERVATIONS:
    raise ValueError(
        "Unexpected observation count for the authenticated source coverage: "
        f"expected {EXPECTED_OBSERVATIONS}, "
        f"observed {len(canonical_period_index)}."
    )

if not canonical_period_index.equals(expected_period_index):
    raise ValueError(
        "The reconstructed canonical period index does not match the "
        "authenticated monthly coverage from 1920-01 through 1939-12."
    )

canonical_source_series = pd.Series(
    source_data["value"].to_numpy(copy=True),
    index=canonical_period_index,
    name="value",
)

temporal_coverage = pd.Series(
    {
        "canonical_index_type": "pandas.PeriodIndex",
        "frequency": "monthly",
        "pandas_frequency": canonical_period_index.freqstr,
        "source_periods_per_year": SOURCE_PERIODS_PER_YEAR,
        "start_period": str(canonical_period_index[0]),
        "end_period": str(canonical_period_index[-1]),
        "observation_count": len(canonical_period_index),
        "coverage_years": 20,
        "timezone": "not applicable to monthly PeriodIndex",
        "max_fractional_year_reconstruction_residual": (
            max_coordinate_residual
        ),
    },
    name="temporal coverage",
)

display(temporal_coverage.to_frame())
display(canonical_source_series.head().to_frame())
display(canonical_source_series.tail().to_frame())

## 5. Target and Univariate Forecasting Contract

The forecasting target is canonically named **`temperature`** and represents the **monthly average air temperature at Nottingham Castle**, measured in **degrees Fahrenheit**. The acquired source column remains `value`; assigning the semantic target name does not alter, transform, aggregate, or rescale any observation.

This study freezes the task as:

- `problem_type`: `time_series_forecasting`;
- `forecasting_mode`: `univariate`;
- one endogenous target series: `temperature`;
- canonical temporal identity: monthly `pandas.PeriodIndex`;
- source frequency: 12 observations per year (`M`);
- source exogenous predictors: none; and
- forecast scale: the original temperature scale in degrees Fahrenheit.

For this study, **univariate** means that the quantity being forecast is a single endogenous series. The deterministic future calendar index implied by the monthly frequency is part of the temporal contract, not an exogenous predictor. No source-provided future covariate series exists.

The forecasting objective is to estimate future values of `temperature` after a forecast origin using only information permitted by the eventual historical-data contract. Forecasted values must remain associated with explicit future monthly periods and preserve the target unit unless a later modeling transformation is explicitly introduced and correctly inverted for public output.

The following concepts are deliberately **not frozen here** because target identity and forecasting geometry are separate concerns:

- forecast horizon;
- forecast origin;
- minimum or maximum allowed history;
- expanding-window or rolling-window evaluation design;
- final-holdout boundary;
- recursive, direct, or other multi-step forecasting strategy;
- baseline definition;
- model family;
- target transformation or differencing policy;
- primary and secondary evaluation metrics; and
- prediction-interval or uncertainty semantics.

Those decisions require evidence from the remaining exploration and the later preparation/evaluation-design notebook. In particular, monthly frequency does not imply that the forecast horizon must be 12 months.

The reusable target-contract helper validates only the structural invariants justified at this point: a numeric endogenous `pandas.Series`, a canonical `PeriodIndex`, the declared frequency, the forecasting problem family, and its univariate mode. Missingness, non-finite values, duplicates, distribution, range, anomalies, seasonal behavior, and forecastability remain dedicated downstream checks.

In [ ]:
from scripts.target_contract import define_univariate_forecasting_target_contract


TARGET_NAME = "temperature"
TARGET_SEMANTICS = "Monthly average air temperature at Nottingham Castle"
TARGET_UNIT = "degrees Fahrenheit"
PROBLEM_TYPE = "time_series_forecasting"
FORECASTING_MODE = "univariate"

forecasting_target_contract = define_univariate_forecasting_target_contract(
    canonical_source_series,
    target=TARGET_NAME,
    problem_type=PROBLEM_TYPE,
    forecasting_mode=FORECASTING_MODE,
    target_semantics=TARGET_SEMANTICS,
    target_unit=TARGET_UNIT,
    expected_frequency=EXPECTED_FREQUENCY,
    source_exogenous_predictors=0,
)

target_series = canonical_source_series.rename(TARGET_NAME)

if not target_series.index.equals(canonical_source_series.index):
    raise ValueError("Canonical target naming must not alter the temporal index.")

if not np.array_equal(
    target_series.to_numpy(),
    canonical_source_series.to_numpy(),
    equal_nan=True,
):
    raise ValueError("Canonical target naming must not alter source values.")

open_forecasting_decisions = pd.Series(
    {
        "forecast_horizon": "open",
        "forecast_origin": "open",
        "allowed_history": "open",
        "backtesting_design": "open",
        "final_holdout_boundary": "open",
        "multi_step_strategy": "open",
        "baseline": "open",
        "model_family": "open",
        "target_transformation": "open",
        "evaluation_metrics": "open",
        "uncertainty_semantics": "open",
    },
    name="status",
)

display(forecasting_target_contract.summary_frame())
display(open_forecasting_decisions.to_frame())
display(target_series.head().to_frame())

## 6. Dataset Structure, Data Types, Units, and Domain Validity

The acquired source has two **transport columns** and 240 rows: `time` carries the original R time-series coordinate and `value` carries the observed temperature measurement. After the canonical reconstruction performed above, the analytical object for forecasting is not a two-feature table but a single numeric `pandas.Series` named `temperature`, indexed by monthly `PeriodIndex` values.

This distinction prevents the physical acquisition format from leaking into the modeling semantics:

- `time` is a numeric source coordinate used to authenticate and reconstruct temporal identity; it is **not** an ordinary predictor;
- `value` is numeric source data and becomes the canonical endogenous target `temperature` without changing its values;
- the analytical dataset contains one endogenous series and zero source exogenous predictors;
- the canonical target index is monthly (`M`) and contains 240 observation periods; and
- no categorical, text, identifier, or source-provided predictor fields belong to the forecasting problem.

The authenticated source unit is **degrees Fahrenheit**. This unit is part of the target contract and must remain attached to observed and forecast values on the public/original scale. No conversion to Celsius, normalization, standardization, or other unit/scale transformation is performed in this exploratory step.

Domain validation is intentionally conservative. For finite observed values, the only universal physical bound enforced here is that an absolute temperature expressed in degrees Fahrenheit cannot be below **absolute zero (`-459.67 °F`)**. No empirical Nottingham-specific lower or upper threshold is imposed, because deriving admissibility limits from the observed historical range would incorrectly turn sample behavior into a source-domain rule.

This section does not decide whether missing or non-finite observations exist, quantify the observed target range, classify extreme values as anomalies, or infer climate-specific outliers. Those questions belong to the dedicated missing/invalid-value, distribution, and anomaly sections that follow.

In [ ]:
ABSOLUTE_ZERO_F = -459.67
EXPECTED_RAW_COLUMNS = ("time", "value")

if tuple(source_data.columns) != EXPECTED_RAW_COLUMNS:
    raise ValueError(
        "Unexpected raw transport structure: "
        f"expected {EXPECTED_RAW_COLUMNS}, "
        f"observed {tuple(source_data.columns)}."
    )

if source_data.shape != (EXPECTED_OBSERVATIONS, len(EXPECTED_RAW_COLUMNS)):
    raise ValueError(
        "Unexpected raw dataset shape: "
        f"expected {(EXPECTED_OBSERVATIONS, len(EXPECTED_RAW_COLUMNS))}, "
        f"observed {source_data.shape}."
    )

if not pd.api.types.is_numeric_dtype(source_data["time"]):
    raise TypeError("The raw source time coordinate must be numeric.")

if not pd.api.types.is_numeric_dtype(source_data["value"]):
    raise TypeError("The raw source value column must be numeric.")

if not isinstance(target_series, pd.Series):
    raise TypeError("The canonical forecasting target must be a pandas Series.")

if target_series.name != TARGET_NAME:
    raise ValueError(
        f"Expected canonical target name {TARGET_NAME!r}, "
        f"observed {target_series.name!r}."
    )

if len(target_series) != EXPECTED_OBSERVATIONS:
    raise ValueError(
        "Unexpected canonical target length: "
        f"expected {EXPECTED_OBSERVATIONS}, observed {len(target_series)}."
    )

if not pd.api.types.is_numeric_dtype(target_series.dtype):
    raise TypeError("The canonical forecasting target must remain numeric.")

if not isinstance(target_series.index, pd.PeriodIndex):
    raise TypeError("The canonical forecasting target must use a PeriodIndex.")

if target_series.index.freqstr != EXPECTED_FREQUENCY:
    raise ValueError(
        "Unexpected canonical target frequency: "
        f"expected {EXPECTED_FREQUENCY!r}, "
        f"observed {target_series.index.freqstr!r}."
    )

if forecasting_target_contract.target_unit != TARGET_UNIT:
    raise ValueError(
        "Target unit changed after contract definition: "
        f"expected {TARGET_UNIT!r}, "
        f"observed {forecasting_target_contract.target_unit!r}."
    )

target_numeric = target_series.to_numpy(dtype=float, copy=True)
finite_target_mask = np.isfinite(target_numeric)
below_absolute_zero_mask = (
    finite_target_mask & (target_numeric < ABSOLUTE_ZERO_F)
)
absolute_zero_violation_count = int(below_absolute_zero_mask.sum())

if absolute_zero_violation_count:
    raise ValueError(
        "Finite target values violate the universal Fahrenheit domain: "
        f"{absolute_zero_violation_count} observation(s) are below "
        f"absolute zero ({ABSOLUTE_ZERO_F} °F)."
    )

dataset_structure = pd.DataFrame(
    [
        ("Raw observations", len(source_data), "Source transport rows"),
        (
            "Raw transport columns",
            len(source_data.columns),
            "time coordinate + observed value",
        ),
        (
            "Raw time dtype",
            str(source_data["time"].dtype),
            "Numeric fractional-year source coordinate",
        ),
        (
            "Raw value dtype",
            str(source_data["value"].dtype),
            "Numeric source temperature value",
        ),
        (
            "Canonical analytical object",
            type(target_series).__name__,
            "One endogenous time series",
        ),
        (
            "Canonical target",
            target_series.name,
            TARGET_SEMANTICS,
        ),
        (
            "Canonical target dtype",
            str(target_series.dtype),
            "Numeric",
        ),
        (
            "Canonical index type",
            type(target_series.index).__name__,
            "Calendar-period identity",
        ),
        (
            "Canonical frequency",
            target_series.index.freqstr,
            "Monthly",
        ),
        (
            "Source exogenous predictors",
            forecasting_target_contract.source_exogenous_predictors,
            "None",
        ),
        (
            "Target unit",
            forecasting_target_contract.target_unit,
            "Original/public target scale",
        ),
        (
            "Finite-value physical domain",
            f">= {ABSOLUTE_ZERO_F} °F",
            "Universal absolute-temperature lower bound",
        ),
        (
            "Physical-domain violations",
            absolute_zero_violation_count,
            "Finite values below absolute zero",
        ),
    ],
    columns=["Property", "Observed", "Interpretation"],
)

display(dataset_structure)

## 7. Temporal Ordering, Missing Periods, and Timestamp Integrity

The canonical time index must represent one and only one observation period for every expected calendar month, in source order, with no temporal reordering or silent gap repair. The reconstruction validated in Section 4 already establishes the expected monthly grid from `1920-01` through `1939-12`; this section makes the corresponding integrity conditions explicit and fail-closed.

Temporal validity requires all of the following:

- the raw fractional-year source coordinate is strictly increasing in row order;
- the canonical `PeriodIndex` is strictly chronological and unique;
- the canonical target index remains exactly aligned with the reconstructed source index;
- consecutive canonical periods advance by exactly one month;
- no expected month is absent from the authenticated 1920–1939 coverage;
- no unexpected period exists outside that coverage; and
- no duplicate canonical period is present.

These checks are structural. They do **not** inspect whether target values are missing or non-finite; that is the responsibility of the next section. Likewise, the absence of duplicate canonical periods is established here as a timestamp-integrity invariant, while repeated temperature values and the scientific distinction between repeated measurements and source revisions remain for the dedicated duplicate/revision analysis.

Because the canonical index is a monthly `PeriodIndex`, timestamp integrity does not require a day, clock time, UTC offset, or timezone. Adding any of those would introduce temporal precision that is not present in the source semantics.

No missing period is imputed or synthesized by this check. If a future acquisition deviates from the authenticated monthly grid, the notebook must fail rather than silently repair the sequence.

In [ ]:
raw_time_numeric = source_data["time"].to_numpy(dtype=float, copy=True)

raw_time_step = np.diff(raw_time_numeric)
raw_time_strictly_increasing = bool(np.all(raw_time_step > 0))

canonical_index = target_series.index

canonical_index_monotonic = bool(canonical_index.is_monotonic_increasing)
canonical_index_unique = bool(canonical_index.is_unique)

duplicate_period_mask = canonical_index.duplicated(keep=False)
duplicate_periods = canonical_index[duplicate_period_mask].unique()

full_expected_index = pd.period_range(
    start=EXPECTED_START_PERIOD,
    end=EXPECTED_END_PERIOD,
    freq=EXPECTED_FREQUENCY,
    name=canonical_index.name,
)

missing_periods = full_expected_index.difference(canonical_index)
unexpected_periods = canonical_index.difference(full_expected_index)

canonical_month_steps = np.diff(canonical_index.asi8)
monthly_step_integrity = bool(
    len(canonical_month_steps) == max(len(canonical_index) - 1, 0)
    and np.all(canonical_month_steps == 1)
)

target_index_aligned = bool(
    canonical_index.equals(canonical_source_series.index)
)

temporal_integrity_failures = []

if not raw_time_strictly_increasing:
    temporal_integrity_failures.append(
        "raw source time coordinate is not strictly increasing"
    )

if not canonical_index_monotonic:
    temporal_integrity_failures.append(
        "canonical PeriodIndex is not monotonically increasing"
    )

if not canonical_index_unique:
    temporal_integrity_failures.append(
        "canonical PeriodIndex contains duplicate periods"
    )

if not monthly_step_integrity:
    temporal_integrity_failures.append(
        "canonical PeriodIndex does not advance by exactly one month"
    )

if len(missing_periods):
    temporal_integrity_failures.append(
        f"{len(missing_periods)} expected monthly period(s) are missing"
    )

if len(unexpected_periods):
    temporal_integrity_failures.append(
        f"{len(unexpected_periods)} unexpected period(s) are present"
    )

if not target_index_aligned:
    temporal_integrity_failures.append(
        "canonical target index is not aligned with the reconstructed source index"
    )

if len(canonical_index) != EXPECTED_OBSERVATIONS:
    temporal_integrity_failures.append(
        "canonical observation count differs from the authenticated source count"
    )

if canonical_index[0] != EXPECTED_START_PERIOD:
    temporal_integrity_failures.append(
        "canonical start period differs from the authenticated source start"
    )

if canonical_index[-1] != EXPECTED_END_PERIOD:
    temporal_integrity_failures.append(
        "canonical end period differs from the authenticated source end"
    )

if temporal_integrity_failures:
    raise ValueError(
        "Temporal integrity validation failed: "
        + "; ".join(temporal_integrity_failures)
        + "."
    )

temporal_integrity_summary = pd.DataFrame(
    [
        (
            "Raw source order",
            raw_time_strictly_increasing,
            "Strictly increasing fractional-year coordinate",
        ),
        (
            "Canonical order",
            canonical_index_monotonic,
            "Monthly periods increase chronologically",
        ),
        (
            "Canonical uniqueness",
            canonical_index_unique,
            "Exactly one canonical observation per period",
        ),
        (
            "One-month step integrity",
            monthly_step_integrity,
            "Every adjacent canonical period advances by one month",
        ),
        (
            "Missing expected periods",
            len(missing_periods),
            "Expected monthly periods absent from canonical coverage",
        ),
        (
            "Unexpected periods",
            len(unexpected_periods),
            "Canonical periods outside authenticated coverage",
        ),
        (
            "Duplicate canonical periods",
            len(duplicate_periods),
            "Repeated monthly identities",
        ),
        (
            "Target/index alignment",
            target_index_aligned,
            "Target preserves reconstructed temporal identity",
        ),
        (
            "Canonical start",
            str(canonical_index[0]),
            f"Expected {EXPECTED_START_PERIOD}",
        ),
        (
            "Canonical end",
            str(canonical_index[-1]),
            f"Expected {EXPECTED_END_PERIOD}",
        ),
        (
            "Canonical observations",
            len(canonical_index),
            f"Expected {EXPECTED_OBSERVATIONS}",
        ),
        (
            "Timezone applicability",
            "not applicable",
            "Monthly PeriodIndex represents calendar periods, not instants",
        ),
    ],
    columns=["Check", "Observed", "Interpretation"],
)

display(temporal_integrity_summary)

## 8. Missing, Invalid, and Non-Finite Values

Temporal completeness and value completeness are separate properties. Section 7 established that the canonical index contains every expected month from `1920-01` through `1939-12`; this does not by itself prove that every monthly period contains a usable temperature measurement.

This section therefore evaluates the canonical endogenous target independently of timestamp integrity. For `temperature`:

- a **missing value** is a canonical monthly observation whose target measurement is absent (`NaN`/missing);
- a **non-finite value** is positive or negative infinity and cannot be treated as an ordinary temperature measurement;
- a **non-numeric value** would violate the numeric target contract established earlier; and
- a **physically invalid finite value** is one below the universal Fahrenheit absolute-zero bound (`-459.67 °F`).

The existing reusable value-quality helper is applied directly to the one-column canonical target frame with explicit rules for required, numeric, finite, and minimum-domain behavior. This validation is intentionally independent of the inherited UCI/Concrete metadata path.

Exploration remains non-mutating. Any detected quality issue is reported as evidence rather than repaired in place: this section does not impute, interpolate, forward-fill, backward-fill, replace, clip, remove, smooth, or otherwise modify target values. If remediation is required, its temporal leakage implications and fold-local behavior must be defined later in the preparation and backtesting design.

Observed historical range, distributional shape, unusual but finite values, and anomaly classification are not used as validity rules here. Those questions remain for the dedicated distribution and anomaly sections.

In [ ]:
from scripts.validate_values import analyze_missing_and_invalid_values


target_value_frame = target_series.to_frame()

target_value_rules = {
    TARGET_NAME: {
        "required": True,
        "allow_blank": False,
        "numeric": True,
        "finite": True,
        "minimum": ABSOLUTE_ZERO_F,
    }
}

value_quality_report = analyze_missing_and_invalid_values(
    target_value_frame,
    target_value_rules,
)

# In exploration, validate rule coverage but retain discovered issues as evidence
# rather than failing before a preparation policy can be designed.
value_quality_report.raise_if_invalid(
    require_all_columns_assessed=True,
    require_rule_columns_present=True,
    require_no_quality_issues=False,
)

target_values = target_series.to_numpy(dtype=float, copy=True)
missing_value_mask = target_series.isna().to_numpy()
finite_value_mask = np.isfinite(target_values)
non_finite_non_missing_mask = (~finite_value_mask) & (~missing_value_mask)
below_absolute_zero_mask = (
    finite_value_mask & (target_values < ABSOLUTE_ZERO_F)
)

missing_value_count = int(missing_value_mask.sum())
non_finite_non_missing_count = int(non_finite_non_missing_mask.sum())
below_absolute_zero_count = int(below_absolute_zero_mask.sum())

value_quality_status = (
    "review required"
    if value_quality_report.has_issues
    else "valid"
)

value_quality_summary = pd.DataFrame(
    [
        (
            "Canonical monthly periods",
            len(target_series),
            "Expected temporal observations",
        ),
        (
            "Missing target values",
            missing_value_count,
            "Monthly periods with no observed temperature value",
        ),
        (
            "Non-finite non-missing values",
            non_finite_non_missing_count,
            "Positive or negative infinity",
        ),
        (
            "Finite values below absolute zero",
            below_absolute_zero_count,
            f"Physical-domain violations below {ABSOLUTE_ZERO_F} °F",
        ),
        (
            "Columns requiring review",
            len(value_quality_report.affected_columns),
            "Canonical target columns with any declared quality issue",
        ),
        (
            "Overall value-quality status",
            value_quality_status,
            (
                "Issues are evidence for preparation; no value mutation "
                "is performed in exploration"
            ),
        ),
    ],
    columns=["Check", "Observed", "Interpretation"],
)

display(value_quality_summary)
display(value_quality_report.column_frame())

if not value_quality_report.issues_frame().empty:
    display(value_quality_report.issues_with_impacts_frame())

## 9. Duplicate Timestamps, Repeated Values, and Source Revision Semantics

Duplicate temporal identities and repeated target values are different phenomena and must not be conflated.

Section 7 established that the canonical monthly `PeriodIndex` is unique. A **duplicate timestamp** would therefore mean that more than one source observation maps to the same canonical month, violating the temporal observation identity of this study. Such a condition is a data-integrity failure.

A **repeated temperature value**, by contrast, means only that two or more distinct monthly periods contain numerically equal observed temperatures. Equal measurements at different times are scientifically permissible and do not represent duplicate observations. They must not be removed, collapsed, averaged, or deduplicated merely because the target values match.

The current source representation contains only the temporal coordinate and one observed value per month. The official `datasets::nottem` documentation identifies a historical time series and cites its published source, but the dataset object does not expose a revision, vintage, release, or observation-version dimension. Consequently, this study has no material basis for interpreting repeated values as source revisions.

The revision semantics frozen at this stage are therefore conservative:

- one canonical month is one temporal observation identity;
- multiple rows for the same canonical month would be a duplicate-timestamp integrity problem;
- equal target values across different canonical months are valid repeated measurements;
- the current dataset representation exposes no parallel vintages or revisions for one month; and
- changes between future acquisitions must be treated as source/provenance drift to investigate, not silently interpreted as valid revisions.

This section is non-mutating. No row is removed and no repeated value is modified. Distributional concentration, persistence, seasonality, and unusual repeated patterns remain separate exploratory questions.

In [ ]:
raw_time_duplicate_mask = source_data["time"].duplicated(keep=False)
raw_time_duplicate_count = int(raw_time_duplicate_mask.sum())

canonical_duplicate_mask = target_series.index.duplicated(keep=False)
canonical_duplicate_count = int(canonical_duplicate_mask.sum())

if raw_time_duplicate_count:
    raise ValueError(
        "Duplicate raw source time coordinates violate temporal observation identity: "
        f"{raw_time_duplicate_count} row(s) are involved."
    )

if canonical_duplicate_count:
    raise ValueError(
        "Duplicate canonical monthly periods violate temporal observation identity: "
        f"{canonical_duplicate_count} row(s) are involved."
    )

target_value_counts = target_series.value_counts(dropna=False).sort_index()
repeated_value_counts = target_value_counts.loc[target_value_counts > 1]

repeated_value_group_count = int(len(repeated_value_counts))
repeated_value_observation_count = int(repeated_value_counts.sum())
distinct_target_value_count = int(target_series.nunique(dropna=False))

repeated_value_rows = []
for repeated_value, occurrence_count in repeated_value_counts.items():
    matching_periods = target_series.index[
        target_series.eq(repeated_value).to_numpy()
    ]
    repeated_value_rows.append(
        {
            "Repeated value": repeated_value,
            "Occurrence count": int(occurrence_count),
            "Periods": ", ".join(str(period) for period in matching_periods),
        }
    )

repeated_values_detail = pd.DataFrame(
    repeated_value_rows,
    columns=["Repeated value", "Occurrence count", "Periods"],
)

revision_like_metadata_fields = {
    "revision",
    "revision_id",
    "version",
    "vintage",
    "release",
    "release_date",
}
observed_revision_metadata_fields = sorted(
    revision_like_metadata_fields.intersection(source_metadata)
)

source_revision_dimension_exposed = bool(
    observed_revision_metadata_fields
    or any(
        str(column).strip().lower() in revision_like_metadata_fields
        for column in source_data.columns
    )
)

duplicate_and_revision_summary = pd.DataFrame(
    [
        (
            "Raw duplicate time coordinates",
            raw_time_duplicate_count,
            "Must be zero for one source observation per temporal coordinate",
        ),
        (
            "Duplicate canonical periods",
            canonical_duplicate_count,
            "Must be zero for one observation identity per calendar month",
        ),
        (
            "Distinct target values",
            distinct_target_value_count,
            "Distinct observed temperature measurements",
        ),
        (
            "Repeated-value groups",
            repeated_value_group_count,
            "Numerically equal values occurring in multiple distinct months",
        ),
        (
            "Observations in repeated-value groups",
            repeated_value_observation_count,
            "Valid observations; not duplicate rows by value alone",
        ),
        (
            "Source revision dimension exposed",
            source_revision_dimension_exposed,
            (
                "Whether the acquired table or source metadata exposes an explicit "
                "revision/vintage field"
            ),
        ),
        (
            "Revision-like metadata fields",
            (
                ", ".join(observed_revision_metadata_fields)
                if observed_revision_metadata_fields
                else "none"
            ),
            "No revision semantics are inferred when explicit evidence is absent",
        ),
    ],
    columns=["Check", "Observed", "Interpretation"],
)

display(duplicate_and_revision_summary)

if not repeated_values_detail.empty:
    display(repeated_values_detail)

## 10. Target Distribution, Range, and Level Summary

The canonical target has passed the preceding structural and value-quality checks, so its observed historical values can now be summarized without imputation, filtering, clipping, or other preprocessing.

This section characterizes the **marginal distribution and historical level** of `temperature` across the complete 240-month source series. The summary includes count, mean, median, sample standard deviation, quartiles, interquartile range, minimum, maximum, observed range, and selected tail percentiles.

These statistics describe the values observed in the available historical record; they do not define admissible domain limits for future observations. In particular, the historical minimum and maximum must not be promoted to hard validation bounds.

The distribution is also visualized with a histogram. Mean and median are shown only as descriptive level references. No observation is classified as an outlier in this section, and no IQR-, z-score-, percentile-, or tail-based removal rule is introduced.

Because this is a time series, the marginal distribution alone cannot determine whether the process is stable through time. Trend, changing level, seasonality, autocorrelation, structural breaks, and temporal anomalies can make an aggregate distribution misleading if interpreted without chronology. Those properties are evaluated in the subsequent time-series sections.

All values remain on the authenticated original scale: **degrees Fahrenheit**.

In [ ]:
import matplotlib.pyplot as plt


if missing_value_count != 0:
    raise ValueError(
        "Distribution analysis requires the previously validated complete target; "
        f"observed missing values={missing_value_count}."
    )

if non_finite_non_missing_count != 0:
    raise ValueError(
        "Distribution analysis requires finite target values; "
        f"observed non-finite values={non_finite_non_missing_count}."
    )

observed_temperature = target_series.astype(float)

quantile_levels = {
    "p05": 0.05,
    "q1": 0.25,
    "median": 0.50,
    "q3": 0.75,
    "p95": 0.95,
}
target_quantiles = observed_temperature.quantile(
    list(quantile_levels.values())
)

q1 = float(target_quantiles.loc[0.25])
median = float(target_quantiles.loc[0.50])
q3 = float(target_quantiles.loc[0.75])
minimum = float(observed_temperature.min())
maximum = float(observed_temperature.max())

target_distribution_summary = pd.DataFrame(
    [
        ("Count", int(observed_temperature.count()), "months"),
        ("Mean", float(observed_temperature.mean()), TARGET_UNIT),
        ("Median", median, TARGET_UNIT),
        ("Sample standard deviation", float(observed_temperature.std(ddof=1)), TARGET_UNIT),
        ("Minimum", minimum, TARGET_UNIT),
        ("5th percentile", float(target_quantiles.loc[0.05]), TARGET_UNIT),
        ("First quartile (Q1)", q1, TARGET_UNIT),
        ("Third quartile (Q3)", q3, TARGET_UNIT),
        ("95th percentile", float(target_quantiles.loc[0.95]), TARGET_UNIT),
        ("Maximum", maximum, TARGET_UNIT),
        ("Interquartile range", q3 - q1, TARGET_UNIT),
        ("Observed range", maximum - minimum, TARGET_UNIT),
    ],
    columns=["Statistic", "Observed", "Unit"],
)

display(target_distribution_summary)

histogram_edges = np.histogram_bin_edges(
    observed_temperature.to_numpy(),
    bins="fd",
)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(
    observed_temperature.to_numpy(),
    bins=histogram_edges,
    alpha=0.8,
)
ax.axvline(
    observed_temperature.mean(),
    linestyle="--",
    linewidth=1.5,
    label="Mean",
)
ax.axvline(
    median,
    linestyle=":",
    linewidth=1.5,
    label="Median",
)
ax.set_title("Distribution of Monthly Average Temperature")
ax.set_xlabel(f"{TARGET_NAME} ({TARGET_UNIT})")
ax.set_ylabel("Monthly observations")
ax.legend()
plt.tight_layout()
plt.show()

## 11. Time-Series Evolution and Long-Term Trend Signals

The marginal distribution summarized above removes chronology. This section restores the temporal order and examines whether the **level of the series appears to evolve across the 1920–1939 record**.

Three complementary descriptive views are used:

- the complete monthly target series, preserving the observed month-to-month path;
- calendar-year mean temperatures, which summarize level across complete 12-month cycles; and
- a simple linear trend fitted to the 20 annual means, reported only as a descriptive long-term slope.

The annual aggregation is appropriate here because Section 7 established a complete regular monthly grid with exactly 12 observations per year, and Section 8 established that no target values require imputation. Annual means therefore compare equally populated calendar years without filling or dropping observations.

The linear annual-mean slope is **not** a forecasting model, a stationarity test, or proof that the underlying process follows a deterministic linear trend. Serial dependence, seasonality, non-linearity, structural changes, and finite-sample effects can all influence the apparent trajectory. Formal stationarity signals and structural-break diagnostics remain for later sections.

To complement the fitted slope without relying on a single line, the notebook also compares the average annual level during the first five years (`1920–1924`) with the last five years (`1935–1939`). This endpoint-window comparison is descriptive evidence only and does not establish causality or a permanent climate trend.

All calculations in this section are retrospective exploration of the full historical series. They must not be reused as globally fitted preprocessing or trend features inside future backtesting folds. Any learned detrending or trend component used for forecasting must be estimated only from the history available at each forecasting origin.

In [ ]:
monthly_plot_index = target_series.index.to_timestamp(how="start")

annual_mean_temperature = (
    target_series
    .groupby(target_series.index.year)
    .mean()
    .astype(float)
)
annual_mean_temperature.index.name = "year"

if len(annual_mean_temperature) != 20:
    raise ValueError(
        "Long-term trend summary expects 20 complete calendar years; "
        f"observed {len(annual_mean_temperature)}."
    )

annual_observation_counts = target_series.groupby(target_series.index.year).count()

if not bool((annual_observation_counts == SOURCE_PERIODS_PER_YEAR).all()):
    raise ValueError(
        "Annual trend summaries require exactly 12 observed monthly values "
        "for every calendar year."
    )

annual_years = annual_mean_temperature.index.to_numpy(dtype=float)
annual_values = annual_mean_temperature.to_numpy(dtype=float)

annual_trend_slope, annual_trend_intercept = np.polyfit(
    annual_years,
    annual_values,
    deg=1,
)
annual_linear_trend = pd.Series(
    annual_trend_intercept + annual_trend_slope * annual_years,
    index=annual_mean_temperature.index,
    name="linear_trend",
)

first_five_year_mean = float(annual_mean_temperature.iloc[:5].mean())
last_five_year_mean = float(annual_mean_temperature.iloc[-5:].mean())
endpoint_window_difference = last_five_year_mean - first_five_year_mean
fitted_record_change = float(
    annual_linear_trend.iloc[-1] - annual_linear_trend.iloc[0]
)

long_term_level_summary = pd.DataFrame(
    [
        (
            "Annual periods summarized",
            len(annual_mean_temperature),
            "complete calendar years",
        ),
        (
            "Months per annual mean",
            SOURCE_PERIODS_PER_YEAR,
            "observations",
        ),
        (
            "Mean annual level, 1920–1924",
            first_five_year_mean,
            TARGET_UNIT,
        ),
        (
            "Mean annual level, 1935–1939",
            last_five_year_mean,
            TARGET_UNIT,
        ),
        (
            "Last-five minus first-five level",
            endpoint_window_difference,
            TARGET_UNIT,
        ),
        (
            "Linear slope across annual means",
            float(annual_trend_slope),
            f"{TARGET_UNIT} per year",
        ),
        (
            "Fitted linear change, 1920–1939",
            fitted_record_change,
            TARGET_UNIT,
        ),
        (
            "Lowest annual mean year",
            int(annual_mean_temperature.idxmin()),
            "calendar year",
        ),
        (
            "Highest annual mean year",
            int(annual_mean_temperature.idxmax()),
            "calendar year",
        ),
    ],
    columns=["Signal", "Observed", "Unit / interpretation"],
)

display(long_term_level_summary)

annual_plot_index = pd.PeriodIndex(
    annual_mean_temperature.index.astype(str),
    freq="Y",
).to_timestamp(how="start")

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(
    monthly_plot_index,
    target_series.to_numpy(dtype=float),
    linewidth=1.0,
    alpha=0.6,
    label="Monthly temperature",
)
ax.plot(
    annual_plot_index,
    annual_mean_temperature.to_numpy(),
    marker="o",
    linewidth=1.8,
    label="Annual mean",
)
ax.plot(
    annual_plot_index,
    annual_linear_trend.to_numpy(),
    linestyle="--",
    linewidth=1.6,
    label="Linear signal across annual means",
)
ax.set_title("Monthly Temperature Evolution and Annual-Level Trend Signal")
ax.set_xlabel("Calendar year")
ax.set_ylabel(f"{TARGET_NAME} ({TARGET_UNIT})")
ax.legend()
plt.tight_layout()
plt.show()

## 12. Seasonal Structure and Calendar-Month Profiles

The source frequency establishes a 12-month calendar cycle, but frequency alone does not prove that the target contains a stable seasonal signal. This section therefore compares the same calendar months across the 20 complete years of observation.

Because temporal integrity and value completeness have already been validated, every calendar month contributes exactly 20 observed temperatures to its month-of-year profile. No interpolation, imputation, resampling, detrending, seasonal adjustment, or transformation is required for this descriptive comparison.

For each calendar month, the notebook summarizes:

- observation count;
- mean temperature;
- median temperature;
- sample standard deviation;
- interquartile range;
- minimum; and
- maximum.

The month with the highest cross-year mean and the month with the lowest cross-year mean are also identified. Their difference is reported as the **calendar-month mean range**. This is a descriptive seasonal amplitude signal only; it is not the formal seasonal-strength statistic evaluated in the decomposition section that follows.

A year-by-month profile plot is used to examine whether a broadly similar within-year shape recurs across calendar years while retaining visibility into interannual variation. The cross-year monthly mean is overlaid as a descriptive reference.

Calendar-month summaries use the full historical series only for exploration. They must not be reused as globally fitted seasonal encodings, imputers, adjustments, or forecasting features during later model evaluation. Any learned seasonal component used in forecasting must be estimated strictly from the historical observations available at each forecasting origin.

This section does not yet choose a seasonal baseline, specify seasonal differencing, fit Holt-Winters/ETS or SARIMA models, quantify decomposition-based seasonal strength, or prove that a seasonal forecasting model is required.

In [ ]:
import calendar


calendar_month = pd.Index(
    target_series.index.month,
    name="calendar_month",
)

month_profile_frame = pd.DataFrame(
    {
        TARGET_NAME: target_series.to_numpy(dtype=float),
        "calendar_month": calendar_month.to_numpy(),
        "calendar_year": target_series.index.year,
    },
    index=target_series.index,
)

monthly_group = month_profile_frame.groupby(
    "calendar_month",
    sort=True,
)[TARGET_NAME]

calendar_month_profile = pd.DataFrame(
    {
        "count": monthly_group.count(),
        "mean": monthly_group.mean(),
        "median": monthly_group.median(),
        "std": monthly_group.std(ddof=1),
        "q1": monthly_group.quantile(0.25),
        "q3": monthly_group.quantile(0.75),
        "minimum": monthly_group.min(),
        "maximum": monthly_group.max(),
    }
)

calendar_month_profile["iqr"] = (
    calendar_month_profile["q3"] - calendar_month_profile["q1"]
)
calendar_month_profile["month_name"] = [
    calendar.month_abbr[month]
    for month in calendar_month_profile.index
]

calendar_month_profile = calendar_month_profile[
    [
        "month_name",
        "count",
        "mean",
        "median",
        "std",
        "q1",
        "q3",
        "iqr",
        "minimum",
        "maximum",
    ]
]

expected_years_per_month = int(
    len(target_series) / SOURCE_PERIODS_PER_YEAR
)

if expected_years_per_month != 20:
    raise ValueError(
        "Calendar-month profiling expects 20 complete annual cycles; "
        f"observed {expected_years_per_month}."
    )

if not bool(
    (
        calendar_month_profile["count"]
        == expected_years_per_month
    ).all()
):
    raise ValueError(
        "Each calendar month must contain exactly one observation "
        "from every complete source year."
    )

warmest_month_number = int(
    calendar_month_profile["mean"].idxmax()
)
coolest_month_number = int(
    calendar_month_profile["mean"].idxmin()
)

calendar_month_mean_range = float(
    calendar_month_profile["mean"].max()
    - calendar_month_profile["mean"].min()
)

seasonal_profile_summary = pd.DataFrame(
    [
        (
            "Complete annual cycles",
            expected_years_per_month,
            "years",
        ),
        (
            "Observations per calendar month",
            int(calendar_month_profile["count"].iloc[0]),
            "observations",
        ),
        (
            "Highest mean calendar month",
            calendar.month_name[warmest_month_number],
            "descriptive cross-year mean",
        ),
        (
            "Lowest mean calendar month",
            calendar.month_name[coolest_month_number],
            "descriptive cross-year mean",
        ),
        (
            "Calendar-month mean range",
            calendar_month_mean_range,
            TARGET_UNIT,
        ),
        (
            "Formal seasonal strength",
            "not evaluated here",
            "reserved for decomposition analysis",
        ),
    ],
    columns=["Signal", "Observed", "Unit / interpretation"],
)

display(calendar_month_profile)
display(seasonal_profile_summary)

year_month_matrix = (
    month_profile_frame
    .pivot(
        index="calendar_month",
        columns="calendar_year",
        values=TARGET_NAME,
    )
    .sort_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
for year in year_month_matrix.columns:
    ax.plot(
        year_month_matrix.index,
        year_month_matrix[year].to_numpy(),
        linewidth=0.8,
        alpha=0.25,
    )

ax.plot(
    calendar_month_profile.index,
    calendar_month_profile["mean"].to_numpy(),
    marker="o",
    linewidth=2.2,
    label="Cross-year monthly mean",
)
ax.set_title("Calendar-Month Temperature Profiles Across Years")
ax.set_xlabel("Calendar month")
ax.set_ylabel(f"{TARGET_NAME} ({TARGET_UNIT})")
ax.set_xticks(range(1, SOURCE_PERIODS_PER_YEAR + 1))
ax.set_xticklabels(
    [
        calendar.month_abbr[month]
        for month in range(1, SOURCE_PERIODS_PER_YEAR + 1)
    ]
)
ax.legend()
plt.tight_layout()
plt.show()

## 13. Decomposition and Trend/Seasonal Strength

The calendar-month profiles above show a large and recurrent within-year temperature cycle, but descriptive month averages do not separate long-term level movement from seasonal structure and residual variation. This section therefore applies an **additive STL decomposition** to the complete canonical monthly series.

STL is used here as an exploratory decomposition, not as a forecasting model. The decomposition period is fixed at the authenticated source frequency of **12 months**. An additive representation is appropriate for this diagnostic because the target is measured on an interval scale in degrees Fahrenheit and this study has not established evidence that seasonal variation should scale multiplicatively with the series level.

The decomposition separates the observed series into:

- a smooth trend component;
- a recurring seasonal component; and
- a remainder containing variation not represented by those two components.

The notebook also reports descriptive **trend strength** and **seasonal strength** statistics using variance ratios:

- trend strength = `max(0, 1 - Var(remainder) / Var(trend + remainder))`;
- seasonal strength = `max(0, 1 - Var(remainder) / Var(seasonal + remainder))`.

Both statistics are bounded to `[0, 1]` for interpretation, where values nearer 1 indicate that the corresponding component explains a larger share of variation relative to the remainder. They are descriptive diagnostics rather than hypothesis tests or model-selection scores.

The decomposition is deliberately fitted with `robust=False`. Robust reweighting could downweight unusual observations before the dedicated anomaly analysis has established whether such treatment is justified. Potential anomalies and structural breaks remain explicit subjects of Section 16.

This full-series STL fit is valid only for retrospective exploration. It is **not leakage-safe preprocessing for forecasting evaluation** because observations across the full record contribute to the fitted decomposition. If detrending, seasonal adjustment, or decomposition-derived features are later used by a forecasting model, they must be learned independently inside each allowed historical window or forecasting origin.

No model family, differencing order, seasonal differencing rule, baseline, forecast horizon, or stationarity conclusion is selected in this section.

In [ ]:
from statsmodels.tsa.seasonal import STL


DECOMPOSITION_PERIOD = SOURCE_PERIODS_PER_YEAR

if DECOMPOSITION_PERIOD != 12:
    raise ValueError(
        "The decomposition period must match the authenticated monthly "
        f"frequency; observed {DECOMPOSITION_PERIOD}."
    )

if len(target_series) < 2 * DECOMPOSITION_PERIOD:
    raise ValueError(
        "STL exploration requires at least two complete seasonal cycles."
    )

if missing_value_count != 0 or non_finite_non_missing_count != 0:
    raise ValueError(
        "STL exploration requires the complete finite target validated "
        "in Section 8."
    )

stl_result = STL(
    target_series.to_numpy(dtype=float),
    period=DECOMPOSITION_PERIOD,
    robust=False,
).fit()

decomposition_trend = pd.Series(
    stl_result.trend,
    index=target_series.index,
    name="trend",
)
decomposition_seasonal = pd.Series(
    stl_result.seasonal,
    index=target_series.index,
    name="seasonal",
)
decomposition_remainder = pd.Series(
    stl_result.resid,
    index=target_series.index,
    name="remainder",
)

reconstructed_values = (
    decomposition_trend
    + decomposition_seasonal
    + decomposition_remainder
)

if not np.allclose(
    reconstructed_values.to_numpy(),
    target_series.to_numpy(dtype=float),
    rtol=1e-10,
    atol=1e-10,
):
    raise ValueError(
        "STL components do not reconstruct the observed target within tolerance."
    )


def _variance_strength(
    component: pd.Series,
    remainder: pd.Series,
) -> float:
    remainder_variance = float(
        np.var(remainder.to_numpy(dtype=float), ddof=1)
    )
    combined_variance = float(
        np.var(
            (component + remainder).to_numpy(dtype=float),
            ddof=1,
        )
    )

    if not np.isfinite(combined_variance) or combined_variance <= 0:
        raise ValueError(
            "Component strength requires positive finite comparison variance."
        )

    strength = 1.0 - (remainder_variance / combined_variance)
    return float(np.clip(strength, 0.0, 1.0))


trend_strength = _variance_strength(
    decomposition_trend,
    decomposition_remainder,
)
seasonal_strength = _variance_strength(
    decomposition_seasonal,
    decomposition_remainder,
)

seasonal_component_range = float(
    decomposition_seasonal.max() - decomposition_seasonal.min()
)
trend_component_change = float(
    decomposition_trend.iloc[-1] - decomposition_trend.iloc[0]
)
remainder_standard_deviation = float(
    decomposition_remainder.std(ddof=1)
)

decomposition_summary = pd.DataFrame(
    [
        (
            "Decomposition method",
            "STL additive",
            "retrospective exploratory decomposition",
        ),
        (
            "Seasonal period",
            DECOMPOSITION_PERIOD,
            "months",
        ),
        (
            "Robust reweighting",
            False,
            "disabled before dedicated anomaly analysis",
        ),
        (
            "Trend strength",
            trend_strength,
            "bounded descriptive strength [0, 1]",
        ),
        (
            "Seasonal strength",
            seasonal_strength,
            "bounded descriptive strength [0, 1]",
        ),
        (
            "Seasonal component range",
            seasonal_component_range,
            TARGET_UNIT,
        ),
        (
            "Trend component endpoint change",
            trend_component_change,
            TARGET_UNIT,
        ),
        (
            "Remainder sample standard deviation",
            remainder_standard_deviation,
            TARGET_UNIT,
        ),
    ],
    columns=["Diagnostic", "Observed", "Interpretation / unit"],
)

display(decomposition_summary)

decomposition_plot_index = target_series.index.to_timestamp(how="start")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(
    decomposition_plot_index,
    target_series.to_numpy(dtype=float),
    linewidth=1.0,
)
ax.set_title("Observed Monthly Temperature")
ax.set_xlabel("Calendar year")
ax.set_ylabel(f"{TARGET_NAME} ({TARGET_UNIT})")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(
    decomposition_plot_index,
    decomposition_trend.to_numpy(dtype=float),
    linewidth=1.4,
)
ax.set_title("STL Trend Component")
ax.set_xlabel("Calendar year")
ax.set_ylabel(TARGET_UNIT)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(
    decomposition_plot_index,
    decomposition_seasonal.to_numpy(dtype=float),
    linewidth=1.0,
)
ax.set_title("STL Seasonal Component")
ax.set_xlabel("Calendar year")
ax.set_ylabel(TARGET_UNIT)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(
    decomposition_plot_index,
    decomposition_remainder.to_numpy(dtype=float),
    linewidth=0.9,
)
ax.axhline(0.0, linewidth=1.0)
ax.set_title("STL Remainder")
ax.set_xlabel("Calendar year")
ax.set_ylabel(TARGET_UNIT)
plt.tight_layout()
plt.show()

## 14. Autocorrelation and Lag Dependence

The decomposition above indicates a strong recurring seasonal component, so serial dependence must be examined explicitly rather than inferred from the calendar profile alone. This section evaluates how strongly the current monthly temperature is associated with earlier observations across short and seasonal lags.

The analysis uses three complementary diagnostics:

- the **autocorrelation function (ACF)** of the observed target through 36 months;
- the **partial autocorrelation function (PACF)** of the observed target through 36 months; and
- the ACF of the exploratory STL remainder, used only to examine whether serial structure remains after the full-series trend and seasonal components are removed.

A 36-month window covers three complete annual cycles and allows direct inspection of the authenticated seasonal lags at 12, 24, and 36 months while retaining shorter-lag structure. Selected lags (`1`, `2`, `3`, `6`, `12`, `13`, `24`, and `36`) are summarized numerically for reproducibility.

The ACF measures total linear association between observations separated by a given lag, including indirect dependence through intermediate lags. The PACF estimates the remaining linear association at a lag after accounting for shorter lags. Neither diagnostic is interpreted here as a final AR or MA order-selection rule.

The STL-remainder ACF is explicitly **retrospective exploratory evidence**. Because the STL decomposition was fitted to the complete historical series, its remainder cannot be used as leakage-safe transformed data in later backtesting or model fitting. It is included only to distinguish dependence that is visually explained by the exploratory trend/seasonal decomposition from dependence that remains unexplained.

The 95% confidence intervals reported by the statistical functions are reference diagnostics, not independent hypothesis-test corrections across many lags. A lag whose interval excludes zero is described as an autocorrelation or partial-autocorrelation signal, not as proof that the corresponding lag must appear in the final forecasting model.

No differencing, seasonal differencing, detrending, AR/MA order, baseline, model family, or forecast horizon is selected in this section. Stationarity and transformation signals are evaluated next.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf


MAX_LAG = 36
REFERENCE_ALPHA = 0.05
INSPECTED_LAGS = (1, 2, 3, 6, 12, 13, 24, 36)

if MAX_LAG > len(target_series) // 2 - 1:
    raise ValueError(
        "Requested PACF lag window is too large for the available series."
    )

observed_values = target_series.to_numpy(dtype=float)
remainder_values = decomposition_remainder.to_numpy(dtype=float)

if not np.isfinite(observed_values).all():
    raise ValueError("Observed target must be finite for lag diagnostics.")

if not np.isfinite(remainder_values).all():
    raise ValueError("STL remainder must be finite for lag diagnostics.")

observed_acf, observed_acf_confint = acf(
    observed_values,
    nlags=MAX_LAG,
    alpha=REFERENCE_ALPHA,
    fft=True,
    missing="raise",
)

observed_pacf, observed_pacf_confint = pacf(
    observed_values,
    nlags=MAX_LAG,
    alpha=REFERENCE_ALPHA,
    method="ywm",
)

remainder_acf, remainder_acf_confint = acf(
    remainder_values,
    nlags=MAX_LAG,
    alpha=REFERENCE_ALPHA,
    fft=True,
    missing="raise",
)


def _interval_excludes_zero(interval: np.ndarray) -> bool:
    lower, upper = float(interval[0]), float(interval[1])
    return bool(lower > 0.0 or upper < 0.0)


lag_dependence_rows = []
for lag in INSPECTED_LAGS:
    lag_dependence_rows.append(
        {
            "lag_months": lag,
            "observed_acf": float(observed_acf[lag]),
            "observed_acf_95pct_excludes_zero": _interval_excludes_zero(
                observed_acf_confint[lag]
            ),
            "observed_pacf": float(observed_pacf[lag]),
            "observed_pacf_95pct_excludes_zero": _interval_excludes_zero(
                observed_pacf_confint[lag]
            ),
            "stl_remainder_acf": float(remainder_acf[lag]),
            "remainder_acf_95pct_excludes_zero": _interval_excludes_zero(
                remainder_acf_confint[lag]
            ),
        }
    )

lag_dependence_summary = pd.DataFrame(lag_dependence_rows)

nonzero_lags = np.arange(1, MAX_LAG + 1)
strongest_observed_acf_lag = int(
    nonzero_lags[
        np.argmax(np.abs(observed_acf[1:]))
    ]
)
strongest_observed_pacf_lag = int(
    nonzero_lags[
        np.argmax(np.abs(observed_pacf[1:]))
    ]
)

significant_observed_acf_lags = [
    int(lag)
    for lag in nonzero_lags
    if _interval_excludes_zero(observed_acf_confint[lag])
]
significant_remainder_acf_lags = [
    int(lag)
    for lag in nonzero_lags
    if _interval_excludes_zero(remainder_acf_confint[lag])
]

lag_structure_overview = pd.DataFrame(
    [
        (
            "Maximum lag inspected",
            MAX_LAG,
            "months",
        ),
        (
            "Seasonal lags inspected",
            "12, 24, 36",
            "authenticated annual cycle multiples",
        ),
        (
            "Strongest observed |ACF| lag",
            strongest_observed_acf_lag,
            "months; descriptive within lags 1–36",
        ),
        (
            "Strongest observed |PACF| lag",
            strongest_observed_pacf_lag,
            "months; descriptive within lags 1–36",
        ),
        (
            "Observed ACF lags with 95% interval excluding zero",
            ", ".join(map(str, significant_observed_acf_lags))
            if significant_observed_acf_lags
            else "none",
            "reference signals; no multiple-testing correction",
        ),
        (
            "STL remainder ACF lags with 95% interval excluding zero",
            ", ".join(map(str, significant_remainder_acf_lags))
            if significant_remainder_acf_lags
            else "none",
            "retrospective residual-dependence signals",
        ),
    ],
    columns=["Diagnostic", "Observed", "Interpretation"],
)

display(lag_structure_overview)
display(lag_dependence_summary)

fig, ax = plt.subplots(figsize=(10, 4.5))
plot_acf(
    observed_values,
    ax=ax,
    lags=MAX_LAG,
    alpha=REFERENCE_ALPHA,
    zero=False,
    fft=True,
)
ax.set_title("Observed Temperature Autocorrelation")
ax.set_xlabel("Lag (months)")
ax.set_ylabel("ACF")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4.5))
plot_pacf(
    observed_values,
    ax=ax,
    lags=MAX_LAG,
    alpha=REFERENCE_ALPHA,
    zero=False,
    method="ywm",
)
ax.set_title("Observed Temperature Partial Autocorrelation")
ax.set_xlabel("Lag (months)")
ax.set_ylabel("PACF")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4.5))
plot_acf(
    remainder_values,
    ax=ax,
    lags=MAX_LAG,
    alpha=REFERENCE_ALPHA,
    zero=False,
    fft=True,
)
ax.set_title("STL Remainder Autocorrelation")
ax.set_xlabel("Lag (months)")
ax.set_ylabel("ACF")
plt.tight_layout()
plt.show()

## 15. Stationarity and Transformation/Differencing Signals

Strong seasonal dependence and a weaker long-term level signal have already been observed, but neither visual trend nor autocorrelation alone establishes whether the series should be differenced or otherwise transformed for forecasting. This section therefore evaluates **stationarity signals** rather than freezing a preprocessing rule.

Two complementary statistical tests are used:

- the **Augmented Dickey–Fuller (ADF)** test, whose null hypothesis is that a unit root is present; and
- the **KPSS** test, whose null hypothesis is stationarity around the deterministic terms included in the test.

For the observed level series, both tests include a constant and deterministic trend (`ct`) so that a slowly changing level is not automatically treated as unexplained drift. Candidate differenced representations use a constant-only specification (`c`).

The following representations are compared:

- observed level;
- first difference, `Δ1 y[t] = y[t] - y[t-1]`;
- seasonal difference, `Δ12 y[t] = y[t] - y[t-12]`; and
- combined first + seasonal difference, `Δ1Δ12 y[t]`.

These candidates are diagnostic only. ADF/KPSS outcomes can be affected by deterministic seasonality, lag selection, finite sample size, structural changes, and the specific regression specification. This section therefore does not translate a p-value mechanically into final ARIMA/SARIMA differencing orders.

For each representation, the notebook also reports sample standard deviation and remaining lag-1 / lag-12 autocorrelation where defined. These descriptive signals help detect whether a candidate difference removes structure, leaves strong dependence, or risks unnecessary over-differencing.

Potential **scale transformations** are treated separately from differencing. Because the target is measured in degrees Fahrenheit, zero on the Fahrenheit scale is arbitrary rather than a physical ratio-scale origin. A logarithm or Box–Cox transformation applied directly to Fahrenheit values would therefore be unit-dependent and is not adopted merely because all observed values are positive. As a descriptive variance-stability signal, the notebook compares annual mean level with annual within-year standard deviation and range, but no power transformation is selected here.

Any transformation, differencing, seasonal adjustment, or learned parameter eventually used for model selection must be performed inside each historical training window. The full-series diagnostics in this section are exploratory evidence only and must not become leakage-producing preprocessing.

No final `d`, `D`, AR/MA order, model family, baseline, forecast horizon, or transformation policy is frozen in this section.

In [ ]:
import warnings

from statsmodels.tsa.stattools import adfuller, kpss


STATIONARITY_ALPHA = 0.05
SEASONAL_DIFFERENCE_LAG = SOURCE_PERIODS_PER_YEAR

stationarity_candidates = {
    "observed_level": target_series.astype(float),
    "first_difference": target_series.diff(1).dropna().astype(float),
    "seasonal_difference_12": (
        target_series.diff(SEASONAL_DIFFERENCE_LAG).dropna().astype(float)
    ),
    "first_plus_seasonal_difference": (
        target_series
        .diff(SEASONAL_DIFFERENCE_LAG)
        .diff(1)
        .dropna()
        .astype(float)
    ),
}

candidate_regressions = {
    "observed_level": "ct",
    "first_difference": "c",
    "seasonal_difference_12": "c",
    "first_plus_seasonal_difference": "c",
}


def _run_adf_test(
    series: pd.Series,
    *,
    regression: str,
) -> dict:
    result = adfuller(
        series.to_numpy(dtype=float),
        regression=regression,
        autolag="AIC",
    )
    statistic, pvalue, used_lag, nobs, critical_values, information_criterion = result

    return {
        "test": "ADF",
        "regression": regression,
        "statistic": float(statistic),
        "p_value": float(pvalue),
        "used_lags": int(used_lag),
        "n_observations": int(nobs),
        "reject_null_at_5pct": bool(pvalue < STATIONARITY_ALPHA),
        "null_hypothesis": "unit root is present",
        "information_criterion": float(information_criterion),
        "critical_value_5pct": float(critical_values["5%"]),
        "warning": "",
    }


def _run_kpss_test(
    series: pd.Series,
    *,
    regression: str,
) -> dict:
    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter("always")
        statistic, pvalue, used_lags, critical_values = kpss(
            series.to_numpy(dtype=float),
            regression=regression,
            nlags="auto",
        )

    warning_text = " | ".join(
        str(item.message).replace("\n", " ")
        for item in caught_warnings
    )

    null_description = (
        "trend stationary"
        if regression == "ct"
        else "level stationary"
    )

    return {
        "test": "KPSS",
        "regression": regression,
        "statistic": float(statistic),
        "p_value": float(pvalue),
        "used_lags": int(used_lags),
        "n_observations": int(len(series)),
        "reject_null_at_5pct": bool(pvalue < STATIONARITY_ALPHA),
        "null_hypothesis": null_description,
        "information_criterion": np.nan,
        "critical_value_5pct": float(critical_values["5%"]),
        "warning": warning_text,
    }


stationarity_test_rows = []

for candidate_name, candidate_series in stationarity_candidates.items():
    if not np.isfinite(candidate_series.to_numpy(dtype=float)).all():
        raise ValueError(
            f"Stationarity candidate {candidate_name!r} contains non-finite values."
        )

    regression = candidate_regressions[candidate_name]

    for test_result in (
        _run_adf_test(candidate_series, regression=regression),
        _run_kpss_test(candidate_series, regression=regression),
    ):
        stationarity_test_rows.append(
            {
                "candidate": candidate_name,
                **test_result,
            }
        )

stationarity_test_summary = pd.DataFrame(stationarity_test_rows)

candidate_signal_rows = []

for candidate_name, candidate_series in stationarity_candidates.items():
    candidate_values = candidate_series.to_numpy(dtype=float)

    candidate_acf = acf(
        candidate_values,
        nlags=min(SEASONAL_DIFFERENCE_LAG, len(candidate_series) - 1),
        fft=True,
        missing="raise",
    )

    lag_1_acf = (
        float(candidate_acf[1])
        if len(candidate_acf) > 1
        else np.nan
    )
    lag_12_acf = (
        float(candidate_acf[SEASONAL_DIFFERENCE_LAG])
        if len(candidate_acf) > SEASONAL_DIFFERENCE_LAG
        else np.nan
    )

    candidate_signal_rows.append(
        {
            "candidate": candidate_name,
            "observations": int(len(candidate_series)),
            "sample_std": float(candidate_series.std(ddof=1)),
            "lag_1_acf": lag_1_acf,
            "lag_12_acf": lag_12_acf,
        }
    )

differencing_signal_summary = pd.DataFrame(candidate_signal_rows)

annual_scale_frame = pd.DataFrame(
    {
        "mean": target_series.groupby(target_series.index.year).mean(),
        "std": target_series.groupby(target_series.index.year).std(ddof=1),
        "range": target_series.groupby(target_series.index.year).max()
        - target_series.groupby(target_series.index.year).min(),
    }
)

if annual_scale_frame.isna().any().any():
    raise ValueError(
        "Annual scale diagnostics require complete within-year observations."
    )

mean_std_correlation = float(
    annual_scale_frame["mean"].corr(annual_scale_frame["std"])
)
mean_range_correlation = float(
    annual_scale_frame["mean"].corr(annual_scale_frame["range"])
)

scale_transformation_signals = pd.DataFrame(
    [
        (
            "Correlation: annual mean vs annual std",
            mean_std_correlation,
            "descriptive level-dispersion association",
        ),
        (
            "Correlation: annual mean vs annual range",
            mean_range_correlation,
            "descriptive level-amplitude association",
        ),
        (
            "Direct log / Box-Cox on Fahrenheit",
            "not selected",
            "Fahrenheit has an arbitrary zero; power-transform choice remains open",
        ),
    ],
    columns=["Signal", "Observed", "Interpretation"],
)

display(stationarity_test_summary)
display(differencing_signal_summary)
display(scale_transformation_signals)

for candidate_name in (
    "first_difference",
    "seasonal_difference_12",
    "first_plus_seasonal_difference",
):
    candidate_series = stationarity_candidates[candidate_name]
    candidate_plot_index = candidate_series.index.to_timestamp(how="start")

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(
        candidate_plot_index,
        candidate_series.to_numpy(dtype=float),
        linewidth=0.9,
    )
    ax.axhline(0.0, linewidth=1.0)
    ax.set_title(
        candidate_name.replace("_", " ").title()
    )
    ax.set_xlabel("Calendar year")
    ax.set_ylabel(f"Difference in {TARGET_UNIT}")
    plt.tight_layout()
    plt.show()

## 16. Outliers, Anomalies, and Structural-Break Signals

An unusual temperature value is not automatically a data error, and an extreme raw observation can be entirely compatible with the regular seasonal cycle. This section therefore distinguishes **point-anomaly candidates** from **persistent structural-change signals** while preserving every source observation.

Point anomalies are evaluated on the exploratory STL remainder rather than on the raw target. This is important because raw winter and summer extremes are expected consequences of the strong annual cycle identified above. The STL decomposition used here was deliberately fitted with `robust=False`, so unusual observations were not downweighted before this analysis.

A robust residual score is calculated from the median and median absolute deviation (MAD):

`robust_score = (remainder - median(remainder)) / (1.4826 × MAD)`

Observations with `|robust_score| >= 3.5` are reported as **anomaly candidates**. This threshold is a screening convention, not proof that an observation is erroneous. Candidate observations remain part of the canonical series and require temporal/domain interpretation before any future remediation could be justified.

Structural stability is examined separately with a deterministic OLS reference model containing:

- an intercept;
- a centered linear time trend; and
- calendar-month indicator terms.

A CUSUM test is then applied to the OLS residuals. Its null hypothesis is parameter stability / no structural change in that reference specification. Because this is a time series with documented serial dependence, the CUSUM p-value is treated only as a **structural-break signal**, not as definitive evidence of a changepoint or a license to segment the dataset.

The period of the largest absolute cumulative-residual excursion is reported only to localize where instability pressure is greatest. It is **not** interpreted as an estimated breakpoint.

This section is entirely retrospective and non-mutating. It does not delete, winsorize, clip, interpolate, replace, or downweight candidate observations. It also does not split the history into regimes, fit a robust forecasting model, or define a structural-break correction.

Both the STL anomaly score and the stability regression use the complete historical record and are therefore exploratory only. If anomaly handling, regime detection, robust fitting, or changepoint logic is later incorporated into model selection, it must be learned independently within each temporally permitted training window.

No preparation rule is frozen here. The decision to preserve, flag, transform, or otherwise handle any candidate anomaly or structural-change signal remains open until the exploratory evidence is consolidated.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.diagnostic import breaks_cusumolsresid


ROBUST_ANOMALY_THRESHOLD = 3.5
STRUCTURAL_BREAK_ALPHA = 0.05

remainder_values = decomposition_remainder.to_numpy(dtype=float)

if not np.isfinite(remainder_values).all():
    raise ValueError(
        "Anomaly screening requires a finite STL remainder."
    )

remainder_center = float(np.median(remainder_values))
remainder_mad = float(
    np.median(np.abs(remainder_values - remainder_center))
)
remainder_robust_scale = 1.4826 * remainder_mad

if not np.isfinite(remainder_robust_scale) or remainder_robust_scale <= 0:
    raise ValueError(
        "Robust anomaly screening requires positive finite MAD-based scale."
    )

remainder_robust_score = pd.Series(
    (remainder_values - remainder_center) / remainder_robust_scale,
    index=target_series.index,
    name="robust_stl_remainder_score",
)

anomaly_candidate_mask = (
    remainder_robust_score.abs() >= ROBUST_ANOMALY_THRESHOLD
)
anomaly_candidate_periods = target_series.index[anomaly_candidate_mask]

anomaly_candidates = pd.DataFrame(
    {
        "temperature": target_series.loc[anomaly_candidate_periods],
        "stl_remainder": decomposition_remainder.loc[anomaly_candidate_periods],
        "robust_remainder_score": remainder_robust_score.loc[
            anomaly_candidate_periods
        ],
    }
)
anomaly_candidates.index.name = "period"

time_position = np.arange(len(target_series), dtype=float)
centered_time_position = time_position - time_position.mean()

calendar_month_dummies = pd.get_dummies(
    target_series.index.month,
    prefix="month",
    drop_first=True,
    dtype=float,
)

stability_design = pd.DataFrame(
    {
        "time_trend": centered_time_position,
    },
    index=target_series.index,
)

calendar_month_dummies.index = target_series.index
stability_design = pd.concat(
    [stability_design, calendar_month_dummies],
    axis=1,
)
stability_design = sm.add_constant(
    stability_design,
    has_constant="add",
)

stability_model = sm.OLS(
    target_series.to_numpy(dtype=float),
    stability_design.to_numpy(dtype=float),
).fit()

stability_residual = pd.Series(
    stability_model.resid,
    index=target_series.index,
    name="seasonal_trend_ols_residual",
)

cusum_statistic, cusum_p_value, cusum_critical_values = (
    breaks_cusumolsresid(
        stability_residual.to_numpy(dtype=float),
        ddof=stability_design.shape[1],
    )
)

cusum_reject_stability = bool(
    cusum_p_value < STRUCTURAL_BREAK_ALPHA
)

residual_scale = float(
    np.sqrt(
        np.sum(stability_residual.to_numpy(dtype=float) ** 2)
        / (len(stability_residual) - stability_design.shape[1])
    )
)

if not np.isfinite(residual_scale) or residual_scale <= 0:
    raise ValueError(
        "CUSUM localization requires positive finite OLS residual scale."
    )

scaled_cumulative_residual = pd.Series(
    np.cumsum(stability_residual.to_numpy(dtype=float))
    / (residual_scale * np.sqrt(len(stability_residual))),
    index=target_series.index,
    name="scaled_cumulative_ols_residual",
)

largest_cusum_excursion_period = (
    scaled_cumulative_residual.abs().idxmax()
)
largest_cusum_excursion = float(
    scaled_cumulative_residual.loc[largest_cusum_excursion_period]
)

structural_break_summary = pd.DataFrame(
    [
        (
            "STL remainder MAD",
            remainder_mad,
            TARGET_UNIT,
        ),
        (
            "MAD-based robust scale",
            remainder_robust_scale,
            TARGET_UNIT,
        ),
        (
            "Anomaly screening threshold",
            ROBUST_ANOMALY_THRESHOLD,
            "|robust remainder score|",
        ),
        (
            "Point-anomaly candidates",
            int(anomaly_candidate_mask.sum()),
            "retained observations requiring review",
        ),
        (
            "CUSUM statistic",
            float(cusum_statistic),
            "OLS parameter-stability reference test",
        ),
        (
            "CUSUM p-value",
            float(cusum_p_value),
            "null: no structural change in reference specification",
        ),
        (
            "Reject CUSUM stability null at 5%",
            cusum_reject_stability,
            "structural-break signal only; serial dependence limits inference",
        ),
        (
            "Largest cumulative-residual excursion period",
            str(largest_cusum_excursion_period),
            "localization signal, not an estimated breakpoint",
        ),
        (
            "Largest signed cumulative-residual excursion",
            largest_cusum_excursion,
            "scaled cumulative OLS residual",
        ),
    ],
    columns=["Diagnostic", "Observed", "Interpretation / unit"],
)

display(structural_break_summary)

if not anomaly_candidates.empty:
    display(
        anomaly_candidates.sort_values(
            "robust_remainder_score",
            key=lambda values: values.abs(),
            ascending=False,
        )
    )

anomaly_plot_index = target_series.index.to_timestamp(how="start")

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(
    anomaly_plot_index,
    target_series.to_numpy(dtype=float),
    linewidth=1.0,
    label="Observed temperature",
)
if len(anomaly_candidate_periods):
    ax.scatter(
        anomaly_candidate_periods.to_timestamp(how="start"),
        target_series.loc[anomaly_candidate_periods].to_numpy(dtype=float),
        marker="o",
        label="STL-remainder anomaly candidate",
    )
ax.set_title("Observed Temperature and Point-Anomaly Candidates")
ax.set_xlabel("Calendar year")
ax.set_ylabel(f"{TARGET_NAME} ({TARGET_UNIT})")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(
    anomaly_plot_index,
    remainder_robust_score.to_numpy(dtype=float),
    linewidth=0.9,
)
ax.axhline(ROBUST_ANOMALY_THRESHOLD, linestyle="--", linewidth=1.0)
ax.axhline(-ROBUST_ANOMALY_THRESHOLD, linestyle="--", linewidth=1.0)
ax.axhline(0.0, linewidth=1.0)
ax.set_title("Robust Scores of STL Remainder")
ax.set_xlabel("Calendar year")
ax.set_ylabel("Robust remainder score")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(
    anomaly_plot_index,
    scaled_cumulative_residual.to_numpy(dtype=float),
    linewidth=1.0,
)
ax.axhline(0.0, linewidth=1.0)
ax.axvline(
    largest_cusum_excursion_period.to_timestamp(how="start"),
    linestyle="--",
    linewidth=1.0,
    label="Largest absolute excursion",
)
ax.set_title("Scaled Cumulative Residual Signal from Seasonal-Trend OLS")
ax.set_xlabel("Calendar year")
ax.set_ylabel("Scaled cumulative residual")
ax.legend()
plt.tight_layout()
plt.show()

## 17. Forecast Horizon and Final-Holdout Feasibility

The forecast horizon must be chosen from the scientific structure of the series and the intended evaluation task, not mechanically from the fact that the data are monthly.

The accumulated evidence now supports a **12-month primary forecast horizon**:

- the source contains 20 complete annual cycles;
- the calendar-month profile shows a large and recurring within-year temperature pattern;
- STL seasonal strength is approximately `0.96`, indicating that the annual seasonal component is dominant relative to the exploratory remainder;
- the strongest observed absolute autocorrelation within the inspected 36-month window occurs at lag 12; and
- a 12-month forecast evaluates predictions across one complete calendar-season cycle rather than only a partial segment of it.

Shorter horizons such as 1, 3, and 6 months remain scientifically meaningful for horizon-wise diagnostics, but they do not by themselves exercise one complete annual cycle. A 24-month horizon is numerically feasible, yet the current evidence does not establish a need to make two full years the primary forecasting task. Using 24 months as the final holdout would also consume twice as much of this small 240-observation series without adding a distinct source-driven forecasting requirement.

The primary forecasting contract is therefore frozen as:

- `forecast_horizon = 12` monthly periods;
- one forecast origin produces 12 ordered future monthly predictions;
- the final forecast origin is `1938-12`;
- the final holdout is `1939-01` through `1939-12`; and
- the pre-holdout history is `1920-01` through `1938-12`, containing 228 observations and 19 complete annual cycles.

This boundary is feasible because it leaves substantially more than one seasonal cycle for model development while reserving one complete, contiguous annual cycle for final evaluation. The final holdout is chronological and immediately follows the authorized development history; there is no random split and no gap between training history and final evaluation.

The term **sealed final holdout** applies prospectively to model development. This exploration notebook has inspected the complete historical source in order to understand the series and design the evaluation boundary, so the 1939 observations are not claimed to be unseen exploratory data. Once this boundary is materialized in the preparation notebook, however, `1939-01` through `1939-12` must be excluded from model selection, hyperparameter selection, transformation fitting, decomposition fitting, lag/feature selection, backtesting, baseline comparison, and any other learned decision. It may be opened only for the one-time final evaluation after the forecasting pipeline is frozen.

The 12-month primary horizon does not imply that all evaluation must be reduced to one aggregate score. Later evaluation may report horizon-wise errors within steps 1–12, provided the metric contract is defined independently and the final holdout remains untouched until finalization.

This section freezes the **forecast horizon and final-holdout boundary only**. It does not yet freeze the rolling/expanding backtesting design, minimum initial history, origin spacing, baseline, metric, model family, or multi-step implementation strategy.

In [ ]:
CANDIDATE_FORECAST_HORIZONS = (1, 3, 6, 12, 24)
SELECTED_FORECAST_HORIZON = 12

FINAL_HOLDOUT_END = target_series.index[-1]
FINAL_HOLDOUT_START = (
    FINAL_HOLDOUT_END - SELECTED_FORECAST_HORIZON + 1
)
FINAL_FORECAST_ORIGIN = FINAL_HOLDOUT_START - 1

development_history = target_series.loc[:FINAL_FORECAST_ORIGIN]
final_holdout_reference = target_series.loc[
    FINAL_HOLDOUT_START:FINAL_HOLDOUT_END
]

if FINAL_HOLDOUT_END != EXPECTED_END_PERIOD:
    raise ValueError(
        "Final-holdout design expects the authenticated source to end at "
        f"{EXPECTED_END_PERIOD}; observed {FINAL_HOLDOUT_END}."
    )

if len(final_holdout_reference) != SELECTED_FORECAST_HORIZON:
    raise ValueError(
        "Final holdout length does not match the selected forecast horizon."
    )

if development_history.index[-1] + 1 != final_holdout_reference.index[0]:
    raise ValueError(
        "Development history and final holdout must be chronologically adjacent."
    )

if not development_history.index.is_monotonic_increasing:
    raise ValueError("Development history must preserve chronological order.")

if not final_holdout_reference.index.is_monotonic_increasing:
    raise ValueError("Final holdout must preserve chronological order.")

candidate_horizon_rows = []

for horizon in CANDIDATE_FORECAST_HORIZONS:
    holdout_start = FINAL_HOLDOUT_END - horizon + 1
    history_end = holdout_start - 1
    history_count = len(target_series.loc[:history_end])
    full_seasonal_cycles = history_count // SOURCE_PERIODS_PER_YEAR

    candidate_horizon_rows.append(
        {
            "horizon_months": horizon,
            "holdout_start": str(holdout_start),
            "holdout_end": str(FINAL_HOLDOUT_END),
            "development_history_observations": history_count,
            "complete_annual_cycles_before_holdout": full_seasonal_cycles,
            "covers_complete_annual_cycle": bool(
                horizon >= SOURCE_PERIODS_PER_YEAR
            ),
            "selected_primary_horizon": bool(
                horizon == SELECTED_FORECAST_HORIZON
            ),
        }
    )

forecast_horizon_feasibility = pd.DataFrame(candidate_horizon_rows)

selected_horizon_summary = pd.DataFrame(
    [
        (
            "Selected primary forecast horizon",
            SELECTED_FORECAST_HORIZON,
            "monthly periods",
        ),
        (
            "Final forecast origin",
            str(FINAL_FORECAST_ORIGIN),
            "last observed period available before final evaluation",
        ),
        (
            "Final holdout start",
            str(FINAL_HOLDOUT_START),
            "first future period in final evaluation",
        ),
        (
            "Final holdout end",
            str(FINAL_HOLDOUT_END),
            "last future period in final evaluation",
        ),
        (
            "Development-history observations",
            len(development_history),
            "months",
        ),
        (
            "Complete annual cycles before final holdout",
            len(development_history) // SOURCE_PERIODS_PER_YEAR,
            "12-month cycles",
        ),
        (
            "Final-holdout observations",
            len(final_holdout_reference),
            "months",
        ),
        (
            "Final holdout seasonal coverage",
            "one complete annual cycle",
            "January through December 1939",
        ),
        (
            "Model-selection holdout status",
            "seal from Notebook 02 onward",
            "exclude from all learned development decisions",
        ),
        (
            "Exploration status",
            "full source already inspected",
            "not claimed to be unseen during exploratory analysis",
        ),
    ],
    columns=["Contract item", "Observed / decision", "Interpretation"],
)

display(forecast_horizon_feasibility)
display(selected_horizon_summary)

## 18. Forecasting Baselines and Forecastability Considerations

A forecasting study requires temporal baselines that respect the information available at each forecast origin. Given the evidence accumulated above, two simple benchmark families are now frozen for later model selection.

The **primary baseline** is **seasonal naive with period 12**:

`ŷ[T+h] = y[T+h-12]`, for `h = 1, ..., 12`.

For the selected 12-month horizon, each forecast period is therefore predicted directly from the observed temperature in the same calendar month one year earlier. No parameter fitting, future information, recursive prediction, smoothing, or learned transformation is required.

The **secondary baseline** is the ordinary **naive / last-observation forecast**:

`ŷ[T+h] = y[T]`, for `h = 1, ..., 12`.

This benchmark ignores seasonality and extends the most recently observed level across the complete forecast horizon. It provides a useful control for determining how much value is contributed simply by honoring the annual cycle.

The seasonal-naive baseline is designated as primary because the exploratory evidence is unusually consistent:

- the authenticated source has a 12-month seasonal period;
- calendar-month profiles show a strong recurring annual shape;
- STL seasonal strength is approximately `0.96`;
- lag 12 is the strongest observed absolute ACF lag within the inspected 36-month window; and
- the 12-month primary forecast horizon allows every seasonal-naive forecast to use an actually observed same-month value from the preceding year.

Forecastability is not synonymous with seasonality, and these signals do not guarantee easy forecasting. The series still contains stochastic remainder variation and residual serial dependence after exploratory seasonal/trend decomposition. The relevant scientific question for later model selection is therefore whether candidate forecasting models can produce **out-of-sample skill beyond seasonal naive** under the frozen temporal evaluation design.

To avoid using the newly defined final holdout for benchmark assessment, this section evaluates forecastability signals only on the `1920-01` through `1938-12` development history. It compares lag-1 and lag-12 persistence and the dispersion of first and seasonal differences. These are descriptive predictability signals, not final evaluation metrics.

No MAE, RMSE, MASE, sMAPE, horizon aggregation rule, or model-selection threshold is frozen here. Likewise, no candidate statistical forecasting model is fitted. Formal baseline scoring and comparison with model families must occur later under the leakage-safe rolling/expanding evaluation design.

The future 1939 forecast periods may be generated from the authorized development history solely to validate baseline **forecast geometry**. Their actual holdout temperatures are not consulted in this section.

The baseline contract frozen here is therefore:

- primary baseline: `seasonal_naive_12`;
- secondary baseline: `naive_last_value`;
- both baselines operate only on history available at the forecast origin;
- both produce 12 ordered monthly forecasts on the original Fahrenheit scale; and
- any more complex model must be evaluated against the seasonal-naive benchmark before additional complexity can be scientifically justified.

In [ ]:
PRIMARY_BASELINE_ID = "seasonal_naive_12"
SECONDARY_BASELINE_ID = "naive_last_value"
BASELINE_SEASONAL_PERIOD = SOURCE_PERIODS_PER_YEAR

if BASELINE_SEASONAL_PERIOD != SELECTED_FORECAST_HORIZON:
    raise ValueError(
        "The primary seasonal-naive baseline expects the selected horizon "
        "to span exactly one authenticated seasonal cycle."
    )

if len(development_history) < BASELINE_SEASONAL_PERIOD:
    raise ValueError(
        "Seasonal-naive forecasting requires at least one complete "
        "12-month historical cycle."
    )

development_values = development_history.to_numpy(dtype=float)

if not np.isfinite(development_values).all():
    raise ValueError(
        "Forecastability diagnostics require finite development-history values."
    )

development_acf = acf(
    development_values,
    nlags=BASELINE_SEASONAL_PERIOD,
    fft=True,
    missing="raise",
)

lag_1_persistence = float(development_acf[1])
lag_12_persistence = float(
    development_acf[BASELINE_SEASONAL_PERIOD]
)

development_level_std = float(
    development_history.std(ddof=1)
)
first_difference_development = (
    development_history.diff(1).dropna()
)
seasonal_difference_development = (
    development_history
    .diff(BASELINE_SEASONAL_PERIOD)
    .dropna()
)

first_difference_std = float(
    first_difference_development.std(ddof=1)
)
seasonal_difference_std = float(
    seasonal_difference_development.std(ddof=1)
)

if development_level_std <= 0:
    raise ValueError(
        "Forecastability diagnostics require positive target variability."
    )

first_difference_variance_ratio = float(
    first_difference_development.var(ddof=1)
    / development_history.var(ddof=1)
)
seasonal_difference_variance_ratio = float(
    seasonal_difference_development.var(ddof=1)
    / development_history.var(ddof=1)
)

future_forecast_index = pd.period_range(
    start=development_history.index[-1] + 1,
    periods=SELECTED_FORECAST_HORIZON,
    freq=EXPECTED_FREQUENCY,
    name=development_history.index.name,
)

seasonal_source_index = (
    future_forecast_index - BASELINE_SEASONAL_PERIOD
)

if not seasonal_source_index.isin(development_history.index).all():
    raise ValueError(
        "Seasonal-naive forecast geometry requires all source periods "
        "to exist in the authorized development history."
    )

naive_baseline_forecast = pd.Series(
    np.repeat(
        float(development_history.iloc[-1]),
        SELECTED_FORECAST_HORIZON,
    ),
    index=future_forecast_index,
    name=SECONDARY_BASELINE_ID,
)

seasonal_naive_baseline_forecast = pd.Series(
    development_history.loc[
        seasonal_source_index
    ].to_numpy(dtype=float),
    index=future_forecast_index,
    name=PRIMARY_BASELINE_ID,
)

baseline_contract = pd.DataFrame(
    [
        (
            PRIMARY_BASELINE_ID,
            "primary",
            "y_hat[T+h] = y[T+h-12]",
            "same calendar month one year earlier",
            "none",
        ),
        (
            SECONDARY_BASELINE_ID,
            "secondary",
            "y_hat[T+h] = y[T]",
            "last observed value at forecast origin",
            "none",
        ),
    ],
    columns=[
        "baseline_id",
        "role",
        "forecast_rule",
        "authorized_history_reference",
        "fitted_parameters",
    ],
)

forecastability_summary = pd.DataFrame(
    [
        (
            "Development-history observations",
            len(development_history),
            "months; final holdout excluded",
        ),
        (
            "Development lag-1 ACF",
            lag_1_persistence,
            "short-lag persistence signal",
        ),
        (
            "Development lag-12 ACF",
            lag_12_persistence,
            "annual recurrence signal",
        ),
        (
            "Development target sample std",
            development_level_std,
            TARGET_UNIT,
        ),
        (
            "First-difference sample std",
            first_difference_std,
            TARGET_UNIT,
        ),
        (
            "Seasonal-difference sample std",
            seasonal_difference_std,
            TARGET_UNIT,
        ),
        (
            "First-difference variance / level variance",
            first_difference_variance_ratio,
            "descriptive relative variability",
        ),
        (
            "Seasonal-difference variance / level variance",
            seasonal_difference_variance_ratio,
            "descriptive relative variability",
        ),
        (
            "Primary benchmark",
            PRIMARY_BASELINE_ID,
            "mandatory later comparison",
        ),
        (
            "Formal baseline performance",
            "not evaluated here",
            "reserved for leakage-safe backtesting",
        ),
    ],
    columns=["Signal", "Observed", "Interpretation / unit"],
)

baseline_forecast_geometry = pd.DataFrame(
    {
        "seasonal_source_period": seasonal_source_index.astype(str),
        SECONDARY_BASELINE_ID: naive_baseline_forecast.to_numpy(),
        PRIMARY_BASELINE_ID: seasonal_naive_baseline_forecast.to_numpy(),
    },
    index=future_forecast_index,
)
baseline_forecast_geometry.index.name = "forecast_period"

display(baseline_contract)
display(forecastability_summary)
display(baseline_forecast_geometry)

## 19. Temporal Leakage and Evaluation-Boundary Risks

Forecasting validity depends on **information availability at each forecast origin**. Chronological ordering alone is not sufficient: a pipeline can preserve row order and still leak future information through globally fitted transformations, decomposition, feature construction, imputation, tuning, or evaluation design.

This study therefore distinguishes three information scopes:

1. **full-source exploratory scope** — the complete `1920-01` through `1939-12` series inspected in Notebook 01 to establish source semantics and exploratory evidence;
2. **model-development scope** — only `1920-01` through `1938-12`, authorized for preparation, backtesting, baseline comparison, tuning, and model selection from Notebook 02 onward; and
3. **sealed final-evaluation scope** — `1939-01` through `1939-12`, reserved prospectively for the one-time final evaluation after the forecasting pipeline is frozen.

A material limitation must be stated explicitly: the 1939 observations are **not exploration-blind**, because this notebook inspected the full historical source before the final boundary was frozen. The final holdout is therefore not claimed to be an external untouched test set in the strongest possible sense. Its role is a **prospectively sealed model-selection holdout**. The mitigation is to freeze the scientific decisions already established in this notebook and prohibit any new learned or selection decision from using 1939 values.

From Notebook 02 onward, the following are prohibited:

- random or shuffled train/validation/test splitting;
- using any 1939 target value during preparation, backtesting, tuning, baseline scoring, model selection, metric selection, lag selection, or transformation selection;
- fitting scalers, imputers, trend estimators, decomposition, seasonal adjustment, Box–Cox/power transforms, or other learned preprocessing on observations beyond the current training origin;
- centered rolling statistics or any feature whose calculation uses future observations;
- constructing lag features without a strictly backward-looking shift;
- selecting hyperparameters or model families from final-holdout performance;
- refitting or retuning after the final holdout has been opened; and
- recomputing alternative final-model candidates after observing final-holdout results.

For rolling-origin or expanding-window evaluation, every validation forecast must obey the same causal rule as production inference: for forecast origin `T`, all fitted state and all model inputs must be functions only of observations at periods `<= T`. The validation targets at `T+1 ... T+12` may be used only to score the already-generated forecast for that origin.

The exploratory STL decomposition, full-series stationarity diagnostics, anomaly screening, and structural-stability analysis in Notebook 01 are **retrospective scientific diagnostics**. Their fitted components or learned quantities must not be carried directly into model selection. Any equivalent learned operation used by a candidate model must be recomputed independently inside each allowed historical window.

The seasonal-naive and naive baselines are naturally causal when implemented from the current historical window. For the 12-month seasonal-naive baseline, every forecast at `T+h` must reference only `y[T+h-12]`, which is already observed at origin `T` for `h = 1, ..., 12`.

The final model workflow must preserve the same boundary discipline:

- Notebook 02 freezes the temporal preparation and backtesting contract while keeping 1939 sealed;
- Notebook 03 performs all baseline comparison, model selection, and tuning only inside the development history;
- Notebook 04 reconstructs the frozen winner, fits it on the authorized pre-holdout history, opens 1939 exactly once for final evaluation, and performs no post-holdout retuning; and
- Notebook 05 consumes only final artifacts for independent forecasting and does not reopen the final holdout or recompute final metrics.

The exact rolling/expanding origin schedule is not frozen in this section. That design belongs to the preparation and backtesting decisions that follow.

In [ ]:
MODEL_DEVELOPMENT_END = FINAL_FORECAST_ORIGIN
SEALED_FINAL_HOLDOUT_START = FINAL_HOLDOUT_START
SEALED_FINAL_HOLDOUT_END = FINAL_HOLDOUT_END

model_development_series = target_series.loc[:MODEL_DEVELOPMENT_END].copy()

sealed_final_holdout_index = pd.period_range(
    start=SEALED_FINAL_HOLDOUT_START,
    end=SEALED_FINAL_HOLDOUT_END,
    freq=EXPECTED_FREQUENCY,
    name=target_series.index.name,
)

if not model_development_series.index.equals(development_history.index):
    raise ValueError(
        "Model-development scope must exactly match the frozen development history."
    )

if len(model_development_series) != 228:
    raise ValueError(
        "Model-development scope must contain exactly 228 monthly observations."
    )

if len(sealed_final_holdout_index) != SELECTED_FORECAST_HORIZON:
    raise ValueError(
        "Sealed final-holdout index must match the selected 12-month horizon."
    )

if model_development_series.index.intersection(
    sealed_final_holdout_index
).size:
    raise ValueError(
        "Model-development history must not overlap the sealed final holdout."
    )

if model_development_series.index[-1] + 1 != sealed_final_holdout_index[0]:
    raise ValueError(
        "The sealed final holdout must begin immediately after development history."
    )

if not future_forecast_index.equals(sealed_final_holdout_index):
    raise ValueError(
        "Baseline forecast geometry must align with the sealed final-holdout periods."
    )

if not seasonal_source_index.isin(model_development_series.index).all():
    raise ValueError(
        "Seasonal-naive source periods must remain inside model-development history."
    )

evaluation_boundary_contract = pd.DataFrame(
    [
        (
            "Full-source exploratory scope",
            "1920-01 -> 1939-12",
            "Notebook 01 retrospective scientific exploration only",
        ),
        (
            "Model-development scope",
            "1920-01 -> 1938-12",
            "authorized for preparation, backtesting, tuning, and selection",
        ),
        (
            "Final forecast origin",
            str(MODEL_DEVELOPMENT_END),
            "latest target observation available before final forecast",
        ),
        (
            "Sealed final-evaluation scope",
            "1939-01 -> 1939-12",
            "one-time final evaluation after model freeze",
        ),
        (
            "Final-holdout length",
            len(sealed_final_holdout_index),
            "monthly periods",
        ),
        (
            "Development / holdout overlap",
            0,
            "required invariant",
        ),
        (
            "Exploration-blind final holdout",
            False,
            "full source was inspected in Notebook 01",
        ),
        (
            "Model-selection sealed from",
            "Notebook 02 onward",
            "no 1939 values may influence new learned or selection decisions",
        ),
    ],
    columns=["Boundary item", "Frozen value", "Interpretation"],
)

leakage_risk_register = pd.DataFrame(
    [
        (
            "Random or shuffled splitting",
            "critical",
            "prohibited",
            "Use chronological rolling/expanding evaluation only",
        ),
        (
            "Final-holdout values used in model development",
            "critical",
            "prohibited",
            "Restrict all learned decisions to <= 1938-12",
        ),
        (
            "Global preprocessing fit before backtesting",
            "critical",
            "prohibited",
            "Fit learned transforms independently inside each training window",
        ),
        (
            "Centered or future-aware rolling features",
            "critical",
            "prohibited",
            "Use strictly backward-looking history available at forecast origin",
        ),
        (
            "Unshifted lag-feature construction",
            "critical",
            "prohibited",
            "Construct lags only from observations known at forecast origin",
        ),
        (
            "Full-series STL/decomposition reused by models",
            "high",
            "prohibited",
            "Notebook 01 decomposition is retrospective diagnostics only",
        ),
        (
            "Metric or model choice after viewing 1939",
            "critical",
            "prohibited",
            "Freeze metrics/models before the final holdout is opened",
        ),
        (
            "Post-final-test retuning",
            "critical",
            "prohibited",
            "No model change after one-time final evaluation",
        ),
        (
            "Exploration exposure to 1939",
            "documented limitation",
            "already occurred",
            "Freeze current design and allow no new 1939-driven decisions",
        ),
        (
            "Seasonal-naive baseline history lookup",
            "controlled",
            "allowed",
            "Require every source period to be <= forecast origin",
        ),
    ],
    columns=["Risk", "Severity", "Policy", "Required control"],
)

display(evaluation_boundary_contract)
display(leakage_risk_register)

## 20. Key Exploratory Insights

The exploratory evidence now supports a coherent scientific picture of `datasets::nottem` as a small, regular, strongly seasonal **univariate monthly forecasting** problem.

The key findings are:

- **Source and temporal identity are clean and explicit.** The canonical series contains 240 monthly observations from `1920-01` through `1939-12`, indexed by a monthly `PeriodIndex`. There are no missing months, duplicate canonical periods, missing target values, non-finite values, or physical-domain violations.
- **The target is a single endogenous temperature series.** `temperature` is measured in degrees Fahrenheit, with no source exogenous predictors. The raw `time` column is a source temporal coordinate, not a tabular predictive feature.
- **The observed level spans a broad seasonal range.** Across the full record, the mean is approximately `49.04 °F`, the median is `47.35 °F`, and the observed values range from `31.3 °F` to `66.5 °F`.
- **Seasonality is the dominant systematic signal.** July has the highest cross-year calendar-month mean and February the lowest, with a calendar-month mean range of `22.71 °F`. STL seasonal strength is approximately `0.960`, substantially larger than trend strength (`≈ 0.241`).
- **Long-term level movement exists but is comparatively modest.** The mean level during `1935–1939` is approximately `0.76 °F` above `1920–1924`, while the descriptive linear slope across annual means is about `0.057 °F` per year. These signals do not establish a deterministic trend model.
- **Lag dependence strongly reflects the annual cycle.** The strongest observed absolute ACF within 36 months occurs at lag 12 (`≈ 0.884` on the full series; `≈ 0.886` within the development history). Short-lag dependence is also substantial, while the exploratory STL remainder still retains some serial structure.
- **Stationarity diagnostics do not justify mechanical differencing.** Under the tested specifications, the observed level rejects the ADF unit-root null while KPSS does not reject trend stationarity. First, seasonal, and combined differences also satisfy their diagnostic tests, but seasonal differencing materially changes the dependence structure. No final `d`, `D`, detrending, or transformation policy is frozen.
- **A direct logarithm or Box–Cox transformation on Fahrenheit is not justified from the current evidence.** Fahrenheit has an arbitrary zero, and the observed association between annual level and annual dispersion is weak. Scale transformation remains open only if a future model-specific rationale requires it.
- **No strong anomaly or structural-break evidence requires intervention.** The robust STL-remainder screen finds zero point-anomaly candidates at the chosen `3.5` threshold, and the reference CUSUM test does not reject stability at 5% (`p ≈ 0.438`). This does not prove perfect stability; it means no current evidence justifies deleting observations or segmenting the history.
- **The primary forecast horizon is 12 months.** The final forecast origin is `1938-12`, leaving `1939-01` through `1939-12` as the prospectively sealed final-evaluation period. Model development is restricted to 228 observations (`1920-01` through `1938-12`), representing 19 complete annual cycles.
- **Forecasting baselines are now defined.** `seasonal_naive_12` is the primary benchmark and `naive_last_value` is secondary. The formal question for later model selection is whether a candidate model can produce leakage-safe out-of-sample skill beyond the seasonal-naive reference.
- **The main methodological risk is temporal leakage.** All future learned transformations, lag construction, decomposition, tuning, and model fitting must use only observations available at each forecast origin. The full-series exploratory diagnostics in this notebook cannot be reused as globally fitted model inputs.

These findings are sufficient to guide the preparation and temporal evaluation design, but they do not yet determine the backtesting origin schedule, minimum initial history, evaluation metrics, candidate model families, multi-step implementation strategy, or uncertainty contract. Those decisions remain for the preparation stage.

In [ ]:
key_exploratory_insights = pd.DataFrame(
    [
        (
            "Temporal coverage",
            f"{target_series.index[0]} -> {target_series.index[-1]}",
            f"{len(target_series)} complete monthly observations",
        ),
        (
            "Value quality",
            "valid",
            (
                f"{missing_value_count} missing; "
                f"{non_finite_non_missing_count} non-finite; "
                f"{below_absolute_zero_count} physical-domain violations"
            ),
        ),
        (
            "Target scale",
            TARGET_UNIT,
            (
                f"mean={observed_temperature.mean():.3f}; "
                f"median={observed_temperature.median():.3f}; "
                f"range={observed_temperature.min():.1f}"
                f"–{observed_temperature.max():.1f}"
            ),
        ),
        (
            "Seasonal structure",
            f"strength={seasonal_strength:.3f}",
            (
                f"{calendar.month_name[coolest_month_number]} lowest mean; "
                f"{calendar.month_name[warmest_month_number]} highest mean; "
                f"calendar-month mean range={calendar_month_mean_range:.2f} °F"
            ),
        ),
        (
            "Trend signal",
            f"strength={trend_strength:.3f}",
            (
                f"annual-mean slope={annual_trend_slope:.3f} °F/year; "
                f"last-five minus first-five={endpoint_window_difference:.3f} °F"
            ),
        ),
        (
            "Lag dependence",
            f"strongest observed |ACF| lag={strongest_observed_acf_lag}",
            (
                f"full-series lag-12 ACF={observed_acf[12]:.3f}; "
                f"development lag-12 ACF={lag_12_persistence:.3f}"
            ),
        ),
        (
            "Stationarity / differencing",
            "no final differencing policy",
            (
                "ADF/KPSS diagnostics support multiple stationary representations; "
                "d/D remain model-selection decisions"
            ),
        ),
        (
            "Anomaly screen",
            int(anomaly_candidate_mask.sum()),
            "point-anomaly candidates retained for review",
        ),
        (
            "Structural stability signal",
            f"CUSUM p={cusum_p_value:.3f}",
            (
                "5% stability null not rejected"
                if not cusum_reject_stability
                else "5% stability null rejected"
            ),
        ),
        (
            "Primary forecast horizon",
            SELECTED_FORECAST_HORIZON,
            "monthly periods / one complete annual cycle",
        ),
        (
            "Development boundary",
            f"{model_development_series.index[0]} -> {model_development_series.index[-1]}",
            f"{len(model_development_series)} observations; final holdout excluded",
        ),
        (
            "Sealed final evaluation",
            f"{SEALED_FINAL_HOLDOUT_START} -> {SEALED_FINAL_HOLDOUT_END}",
            "prospectively sealed from model-selection decisions",
        ),
        (
            "Primary baseline",
            PRIMARY_BASELINE_ID,
            "mandatory benchmark for later leakage-safe model selection",
        ),
        (
            "Secondary baseline",
            SECONDARY_BASELINE_ID,
            "non-seasonal level reference",
        ),
        (
            "Principal evaluation risk",
            "temporal leakage",
            "all learned operations must be fit within each authorized history window",
        ),
    ],
    columns=["Insight", "Frozen evidence / decision", "Interpretation"],
)

display(key_exploratory_insights)

## 21. Preparation and Backtesting Decisions

The exploratory evidence is now sufficient to freeze the preparation and temporal evaluation contract that Notebook 02 must materialize. These decisions are designed specifically for a small, regular, strongly seasonal univariate series with a 12-month primary forecast horizon and a prospectively sealed 1939 final holdout.

### Preparation policy

The canonical target must remain the observed monthly `temperature` series on its original Fahrenheit scale. The source contains no missing periods, missing target values, non-finite values, duplicate timestamps, or anomaly candidates that justify repair. Preparation must therefore preserve all authorized observations without imputation, interpolation, clipping, winsorization, outlier removal, resampling, or synthetic filling.

No transformation is frozen globally. In particular:

- no logarithm or Box–Cox transform is applied to Fahrenheit values;
- no first or seasonal differencing is materialized as a universal prepared target;
- no full-history STL decomposition, detrending, or seasonal adjustment is carried forward as fitted preprocessing;
- no global scaling or normalization is required for the canonical series; and
- no exogenous predictors are introduced.

A candidate model may still require differencing, scaling, decomposition, lag construction, or other learned state internally. Any such operation must be fitted independently from the training history available at each forecast origin and must never use its validation window or the sealed final holdout.

### Backtesting geometry

Model development uses an **expanding-window, rolling-origin evaluation** with non-overlapping 12-month validation windows.

The frozen geometry is:

- model-development history: `1920-01` through `1938-12` (`228` months);
- initial training window: `1920-01` through `1929-12` (`120` months / 10 complete annual cycles);
- first backtest origin: `1929-12`;
- forecast horizon per origin: `12` months;
- origin step: `12` months;
- final backtest origin: `1937-12`;
- final backtest validation window: `1938-01` through `1938-12`;
- total backtest folds: `9`;
- total out-of-sample backtest forecasts: `108`; and
- sealed final evaluation: `1939-01` through `1939-12`, excluded from every backtest fold.

An expanding window is preferred over a fixed rolling window because the dataset is small, earlier observations remain scientifically relevant, and the current structural-stability evidence does not justify discarding older history. A 120-month initial window gives every first-fold candidate 10 complete seasonal cycles while still leaving nine full annual validation blocks for model comparison.

The 12-month origin step deliberately avoids overlapping validation targets. It also reproduces the final deployment geometry: every backtest origin occurs at the end of December and forecasts the following January–December cycle. This design prioritizes direct comparability with the frozen final forecast origin over evaluation across every possible month-of-year origin.

At each fold origin `T`, the candidate must generate the complete ordered forecast `T+1 ... T+12` **without observing any value inside that validation window**. Candidate-specific multi-step implementations may differ, but recursive updating with realized validation targets is prohibited.

### Evaluation metrics

The primary model-selection metric is **MAE** aggregated across all 108 backtest forecast errors. MAE remains directly interpretable in degrees Fahrenheit and gives each forecasted month equal weight.

Two secondary metrics are frozen:

- **RMSE**, also in degrees Fahrenheit, to expose sensitivity to larger forecast errors; and
- **seasonal MASE with `m=12`**, whose scaling denominator must be calculated independently inside each fold from that fold's training history using the mean absolute seasonal difference `|y[t] - y[t-12]|`.

For equal 12-month folds, pooled MAE is numerically equivalent to the equally weighted mean of fold MAEs. Horizon-wise MAE for forecast steps `h=1 ... 12` must also be reported as a diagnostic so that an apparently strong aggregate result cannot hide systematic degradation at longer horizons.

`MAPE` and `sMAPE` are not part of the evaluation contract. Percentage errors are not scientifically attractive for a Fahrenheit target because the Fahrenheit zero is arbitrary; the resulting percentages are not invariant to a change of temperature origin.

The frozen primary and secondary baselines (`seasonal_naive_12` and `naive_last_value`) must be scored under exactly the same folds, forecast horizon, and metrics as every candidate model. Model complexity is justified only by leakage-safe backtest evidence relative to the primary seasonal-naive benchmark.

### Selection and final-evaluation boundary

Notebook 03 may use only the nine frozen backtest folds for baseline comparison, model-family comparison, hyperparameter selection, and any model-specific transformation decisions. No metric, model family, hyperparameter, lag policy, or preprocessing choice may be selected from the 1939 values.

Notebook 04 may fit the already frozen winning specification on the complete `1920-01` through `1938-12` development history and then open the 12-month 1939 holdout exactly once. No post-holdout retuning or alternative winner selection is allowed.

This preparation contract requires **point forecasts** for 12 ordered monthly periods on the original Fahrenheit scale. Forecast intervals or other probabilistic uncertainty outputs are not required for the primary model-selection contract in this first univariate study; they may be considered later only under a separately defined, leakage-safe evaluation contract.

Candidate model families and their internal multi-step forecasting implementations remain intentionally unfrozen until model-selection design.

In [ ]:
BACKTEST_MODE = "expanding_window"
INITIAL_TRAINING_MONTHS = 120
BACKTEST_HORIZON = SELECTED_FORECAST_HORIZON
BACKTEST_STEP_MONTHS = 12
PRIMARY_SELECTION_METRIC = "mae"
SECONDARY_SELECTION_METRICS = ("rmse", "seasonal_mase_12")
MASE_SEASONAL_PERIOD = SOURCE_PERIODS_PER_YEAR

if INITIAL_TRAINING_MONTHS % SOURCE_PERIODS_PER_YEAR != 0:
    raise ValueError(
        "Initial training history must contain an integer number of annual cycles."
    )

if BACKTEST_HORIZON != SOURCE_PERIODS_PER_YEAR:
    raise ValueError(
        "Backtesting horizon must match the frozen 12-month primary horizon."
    )

if BACKTEST_STEP_MONTHS != BACKTEST_HORIZON:
    raise ValueError(
        "This contract requires non-overlapping validation windows."
    )

if len(model_development_series) != 228:
    raise ValueError(
        "Backtesting must operate only on the 228-month model-development history."
    )

origin_positions = list(
    range(
        INITIAL_TRAINING_MONTHS - 1,
        len(model_development_series) - BACKTEST_HORIZON,
        BACKTEST_STEP_MONTHS,
    )
)

backtest_rows = []

for fold_number, origin_position in enumerate(origin_positions, start=1):
    forecast_origin = model_development_series.index[origin_position]
    validation_start = forecast_origin + 1
    validation_end = forecast_origin + BACKTEST_HORIZON

    training_index = model_development_series.index[: origin_position + 1]
    validation_index = pd.period_range(
        start=validation_start,
        end=validation_end,
        freq=EXPECTED_FREQUENCY,
    )

    if len(validation_index) != BACKTEST_HORIZON:
        raise ValueError(
            f"Fold {fold_number} does not contain the required 12-month horizon."
        )

    if training_index[-1] + 1 != validation_index[0]:
        raise ValueError(
            f"Fold {fold_number} training and validation windows are not adjacent."
        )

    if not validation_index.isin(model_development_series.index).all():
        raise ValueError(
            f"Fold {fold_number} validation window escapes development history."
        )

    if validation_index.intersection(sealed_final_holdout_index).size:
        raise ValueError(
            f"Fold {fold_number} overlaps the sealed final holdout."
        )

    seasonal_mase_differences = (
        model_development_series
        .loc[training_index]
        .diff(MASE_SEASONAL_PERIOD)
        .dropna()
        .abs()
    )

    if seasonal_mase_differences.empty:
        raise ValueError(
            f"Fold {fold_number} lacks history for seasonal MASE scaling."
        )

    seasonal_mase_scale = float(
        seasonal_mase_differences.mean()
    )

    if not np.isfinite(seasonal_mase_scale) or seasonal_mase_scale <= 0:
        raise ValueError(
            f"Fold {fold_number} has an invalid seasonal MASE denominator."
        )

    backtest_rows.append(
        {
            "fold": fold_number,
            "train_start": str(training_index[0]),
            "train_end_forecast_origin": str(forecast_origin),
            "training_observations": len(training_index),
            "complete_training_cycles": (
                len(training_index) // SOURCE_PERIODS_PER_YEAR
            ),
            "validation_start": str(validation_index[0]),
            "validation_end": str(validation_index[-1]),
            "validation_observations": len(validation_index),
            "seasonal_mase_scale_from_training": seasonal_mase_scale,
        }
    )

backtesting_schedule = pd.DataFrame(backtest_rows)

if len(backtesting_schedule) != 9:
    raise ValueError(
        "Frozen backtesting geometry must produce exactly 9 folds."
    )

if int(backtesting_schedule["validation_observations"].sum()) != 108:
    raise ValueError(
        "Frozen backtesting geometry must produce exactly 108 validation forecasts."
    )

if backtesting_schedule.iloc[0]["train_end_forecast_origin"] != "1929-12":
    raise ValueError("First backtest origin must be 1929-12.")

if backtesting_schedule.iloc[-1]["train_end_forecast_origin"] != "1937-12":
    raise ValueError("Final backtest origin must be 1937-12.")

if backtesting_schedule.iloc[-1]["validation_end"] != "1938-12":
    raise ValueError(
        "Final backtest validation window must end at 1938-12."
    )

preparation_backtesting_contract = pd.DataFrame(
    [
        (
            "Preparation target",
            TARGET_NAME,
            "canonical univariate monthly series",
        ),
        (
            "Prepared target scale",
            TARGET_UNIT,
            "original scale; no global power transformation",
        ),
        (
            "Missing-value action",
            "none",
            "source target is complete",
        ),
        (
            "Outlier/anomaly action",
            "preserve all observations",
            "no anomaly candidate currently justifies mutation",
        ),
        (
            "Global differencing",
            "none",
            "candidate-specific and fold-local only if required",
        ),
        (
            "Global decomposition / detrending",
            "none",
            "full-series exploratory fits cannot enter model development",
        ),
        (
            "Backtest mode",
            BACKTEST_MODE,
            "expanding historical training window",
        ),
        (
            "Initial training history",
            INITIAL_TRAINING_MONTHS,
            "months / 10 complete annual cycles",
        ),
        (
            "Forecast horizon",
            BACKTEST_HORIZON,
            "months per origin",
        ),
        (
            "Origin step",
            BACKTEST_STEP_MONTHS,
            "months; validation windows do not overlap",
        ),
        (
            "Backtest folds",
            len(backtesting_schedule),
            "1930 through 1938 validation years",
        ),
        (
            "Backtest forecast observations",
            int(backtesting_schedule["validation_observations"].sum()),
            "9 folds × 12 horizons",
        ),
        (
            "Primary selection metric",
            PRIMARY_SELECTION_METRIC,
            "pooled absolute error on original °F scale",
        ),
        (
            "Secondary metrics",
            ", ".join(SECONDARY_SELECTION_METRICS),
            "large-error sensitivity + seasonal scale-free error",
        ),
        (
            "Horizon-wise diagnostic",
            "MAE for h=1..12",
            "required diagnostic; not a separate final-holdout selection rule",
        ),
        (
            "Percentage-error metrics",
            "MAPE/sMAPE excluded",
            "Fahrenheit has an arbitrary zero",
        ),
        (
            "Required point forecast output",
            12,
            "ordered future monthly predictions",
        ),
        (
            "Final holdout",
            "1939-01 -> 1939-12",
            "excluded from preparation and all backtesting",
        ),
    ],
    columns=["Decision", "Frozen value", "Rationale / interpretation"],
)

display(preparation_backtesting_contract)
display(backtesting_schedule)

## 22. Exploration Handoff and Next Steps

Notebook 01 now contains enough validated evidence to publish a machine-readable **exploration handoff** for the preparation stage. The handoff must preserve forecasting semantics rather than inherit the snapshot-split assumptions of the static classification/regression studies.

The artifact written by this section is:

`artifacts/exploration/nottem/exploration-handoff.json`

It uses the existing project-level `exploration-handoff.v1` artifact identity while carrying a forecasting-specific contract with:

- scientific source identity and raw-source SHA-256;
- `time_series_forecasting` / `univariate` prediction semantics;
- canonical monthly `PeriodIndex` coverage and original Fahrenheit target scale;
- zero source exogenous predictors and no tabular feature contract;
- primary forecast horizon of 12 months;
- model-development boundary through `1938-12`;
- prospectively sealed final holdout `1939-01` through `1939-12`;
- expanding-window backtesting with 120 initial training months, 12-month horizon, 12-month step, nine non-overlapping folds, and 108 validation forecasts;
- primary `MAE`, secondary `RMSE` and seasonal `MASE(m=12)`, plus horizon-wise MAE diagnostics;
- primary `seasonal_naive_12` and secondary `naive_last_value` baselines;
- leakage controls and the documented limitation that 1939 was inspected during retrospective Notebook 01 exploration; and
- continuation/readiness gates for Notebook 02.

Forecasting does **not** authorize the inherited static-study snapshot split. The handoff therefore declares `split_execution_ready = false` and `temporal_backtesting_ready = true`. `model_selection_ready` remains false until Notebook 02 independently reconstructs the source, materializes the frozen development/holdout boundary and backtesting schedule, validates temporal causality, and publishes its own preparation handoff.

Three non-blocking decisions remain deliberately open for Notebook 03 rather than being resolved here:

1. candidate forecasting model families;
2. candidate-specific differencing, detrending, decomposition, scaling, or other fold-local transformations; and
3. the internal multi-step strategy used by each candidate to generate all 12 forecasts without observing validation targets.

Forecast intervals are not required by the current point-forecast model-selection contract. The final 1939 holdout must remain unopened for scoring until the winning forecasting specification has been frozen and Notebook 04 performs the one-time final evaluation.

In [ ]:
from scripts.forecasting_exploration_handoff import (
    build_univariate_forecasting_exploration_handoff,
    load_and_validate_forecasting_exploration_handoff,
)

DATASET_SLUG = "nottem"
EXPLORATION_HANDOFF_RELATIVE_PATH = (
    f"artifacts/exploration/{DATASET_SLUG}/exploration-handoff.json"
)

exploration_handoff_path = PROJECT.path(
    "artifacts",
    "exploration",
    DATASET_SLUG,
    "exploration-handoff.json",
)
exploration_handoff_path.resolve().relative_to(PROJECT.root)

exploration_handoff_report = build_univariate_forecasting_exploration_handoff(
    dataset_slug=DATASET_SLUG,
    source_repository=source_metadata["source_archive"],
    source_reference=source_metadata["source_reference"],
    source_dataset_name=source_metadata["dataset"],
    source_package=source_metadata["package"],
    source_provider=source_metadata["provider"],
    source_file=data_path,
    project_root=PROJECT.root,
    source_dataframe=source_data,
    target_contract=forecasting_target_contract,
    canonical_series=target_series,
    forecast_horizon=SELECTED_FORECAST_HORIZON,
    development_history=model_development_series,
    final_forecast_origin=FINAL_FORECAST_ORIGIN,
    final_holdout_start=FINAL_HOLDOUT_START,
    final_holdout_end=FINAL_HOLDOUT_END,
    primary_baseline_id=PRIMARY_BASELINE_ID,
    secondary_baseline_id=SECONDARY_BASELINE_ID,
    backtesting_schedule=backtesting_schedule,
    preparation_backtesting_contract=preparation_backtesting_contract,
    evaluation_boundary_contract=evaluation_boundary_contract,
    leakage_risk_register=leakage_risk_register,
    key_exploratory_insights=key_exploratory_insights,
    primary_metric=PRIMARY_SELECTION_METRIC,
    secondary_metrics=SECONDARY_SELECTION_METRICS,
    mase_seasonal_period=MASE_SEASONAL_PERIOD,
)

exploration_handoff_report.raise_if_invalid()

persisted_exploration_handoff = exploration_handoff_report.write(
    exploration_handoff_path
)
persisted_exploration_handoff.path.resolve().relative_to(PROJECT.root)

reloaded_exploration_handoff = (
    load_and_validate_forecasting_exploration_handoff(
        exploration_handoff_path,
        expected_dataset_slug=DATASET_SLUG,
        expected_source_reference=EXPECTED_SOURCE_REFERENCE,
    )
)

if reloaded_exploration_handoff["prediction_contract"]["forecast_horizon"] != 12:
    raise ValueError("Persisted exploration handoff changed the frozen horizon.")

if reloaded_exploration_handoff["temporal_contract"]["development_end"] != "1938-12":
    raise ValueError("Persisted exploration handoff changed the development boundary.")

if reloaded_exploration_handoff["temporal_contract"]["final_holdout_start"] != "1939-01":
    raise ValueError("Persisted exploration handoff changed the final holdout start.")

if reloaded_exploration_handoff["backtesting_contract"]["fold_count"] != 9:
    raise ValueError("Persisted exploration handoff changed the backtesting fold count.")

persisted_readiness = reloaded_exploration_handoff["readiness"]

if persisted_readiness["split_execution_ready"] is not False:
    raise ValueError("Forecasting handoff must not authorize snapshot split execution.")

if persisted_readiness["temporal_backtesting_ready"] is not True:
    raise ValueError("Forecasting handoff must authorize the frozen temporal backtest.")

if persisted_readiness["model_selection_ready"] is not False:
    raise ValueError("Notebook 01 must not authorize model selection directly.")

handoff_artifact_summary = pd.DataFrame(
    [
        (
            "Artifact path",
            PROJECT.display(persisted_exploration_handoff.path),
        ),
        (
            "Schema version",
            reloaded_exploration_handoff["schema_version"],
        ),
        (
            "SHA-256",
            persisted_exploration_handoff.sha256,
        ),
        (
            "Bytes",
            persisted_exploration_handoff.size_bytes,
        ),
        (
            "Notebook 01 complete",
            persisted_readiness["notebook_01_complete"],
        ),
        (
            "Temporal backtesting ready",
            persisted_readiness["temporal_backtesting_ready"],
        ),
        (
            "Model selection ready",
            persisted_readiness["model_selection_ready"],
        ),
    ],
    columns=["Handoff item", "Observed"],
)

display(exploration_handoff_report.summary_frame())
display(exploration_handoff_report.open_reviews_frame())
display(exploration_handoff_report.next_steps_frame())
display(exploration_handoff_report.expected_outputs_frame())
display(handoff_artifact_summary)